# In-house Fin-JEPA and T-JEPA fraud analysis

Fraud detection on the in-house transaction dataset using two self-supervised anomaly scorers:

- **Fin-JEPA:** sequence-level JEPA over customer transaction windows.
- **T-JEPA:** feature-level JEPA with context/target feature masking.

Both are trained without fraud labels and evaluated on the official test split with ROC-AUC, PR-AUC, Precision@k / Recall@k, precision at fixed recall, and per-typology breakdowns.

Data paths point to the Kaggle input directory (`BASE` / `DATA_DIR`); edit them if you run this elsewhere. Run the notebook top to bottom.

## Part 1: Data preparation and verification

Load the in-house dataset, build training labels, engineer and standardize features, and verify the charge mapping, official split and window logic.

### Environment setup

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# kagglehub.dataset_download('<owner>/<dataset-slug>')

### Load data and build training labels

In [ ]:
# Load the three core tables:
#   sequences      - one row per transaction with features and seq_pos
#   label_timeline - noisy, real-time labels (not ground truth)
#   ground_truth   - true fraud labels, held out for evaluation only

import pandas as pd
import numpy as np

BASE = "/kaggle/input/datasets/abhradwiplala/in-house-dataset"  
sequences = pd.read_csv(f"{BASE}/sequences.csv")
label_timeline = pd.read_csv(f"{BASE}/label_timeline.csv")
ground_truth = pd.read_csv(f"{BASE}/_ground_truth.csv")

# Training label: combine all label channels (simplification -- see note above)
label_timeline['train_label'] = (
    (label_timeline['y_efw'] == 1) | (label_timeline['y_review'] == 1) |
    (label_timeline['y_dispute'] == 1) | (label_timeline['y_refund'] == 1)
).astype(int)

print("Train-time label positive rate:", label_timeline['train_label'].mean())
print("True fraud rate (ground truth):", ground_truth['is_fraud'].mean())
print("(These SHOULD differ -- label_timeline is realistically partial, this gap is expected and is the point of the dataset)")

# Merge everything together, keyed by charge id
df = sequences.merge(label_timeline[['charge_id', 'train_label']], left_on='id', right_on='charge_id', how='left')
df = df.merge(ground_truth, left_on='id', right_on='charge_id', how='left', suffixes=('', '_gt'))

print("\nMerged shape:", df.shape)
print("Train rows:", (df['split']=='train').sum(), "Test rows:", (df['split']=='test').sum())

### Feature preprocessing and Fin-JEPA window construction

In [ ]:
import numpy as np

# ── Feature selection: drop IDs/raw timestamps, keep engineered + descriptive features ──
DROP_COLS = ['id', 'customer_id', 'card_id', 'connected_account_id', 'created', 'payment_method_id',
             'payment_intent_id', 'balance_transaction_id', 'on_behalf_of_id', 'bank_account_id',
             'source_id', 'card_fingerprint', 'card_iin', 'card_issuer', 'cus_created', 'charge_id',
             'charge_id_gt', 'is_fraud', 'typology', 'train_label', 'split']

CATEGORICAL_COLS = ['currency', 'billing_detail_address_country', 'billing_detail_address_city',
                     'card_brand', 'card_funding', 'card_country', 'card_cvc_check',
                     'cus_country', 'cus_city', 'mcc', 'acct_country', 'acct_structure']

NUMERIC_COLS = [c for c in df.columns if c not in DROP_COLS + CATEGORICAL_COLS]
print(f"Numeric: {len(NUMERIC_COLS)}, Categorical: {len(CATEGORICAL_COLS)}")

# ── Encode categoricals (fill missing as its own category first) ──
from sklearn.preprocessing import LabelEncoder
for col in CATEGORICAL_COLS:
    df[col] = df[col].fillna('MISSING').astype(str)
    df[col] = LabelEncoder().fit_transform(df[col])

# ── Impute + standardize numerics using TRAIN stats only, then clip (lesson learned from IEEE-CIS) ──
train_mask = df['split'] == 'train'
medians = df.loc[train_mask, NUMERIC_COLS].median()
df[NUMERIC_COLS] = df[NUMERIC_COLS].fillna(medians)

feat_mean = df.loc[train_mask, NUMERIC_COLS].mean()
# Compute mean/std on TRAIN rows only to prevent data leakage into the test set.

feat_std = df.loc[train_mask, NUMERIC_COLS].std() + 1e-8
df[NUMERIC_COLS] = (df[NUMERIC_COLS] - feat_mean) / feat_std

cat_mean = df.loc[train_mask, CATEGORICAL_COLS].mean()
cat_std = df.loc[train_mask, CATEGORICAL_COLS].std() + 1e-8
df[CATEGORICAL_COLS] = (df[CATEGORICAL_COLS] - cat_mean) / cat_std

CLIP = 10.0
df[NUMERIC_COLS + CATEGORICAL_COLS] = df[NUMERIC_COLS + CATEGORICAL_COLS].clip(-CLIP, CLIP)

MODEL_FEATURES = NUMERIC_COLS + CATEGORICAL_COLS
print(f"Total model features: {len(MODEL_FEATURES)}")
print("Any NaN left?", df[MODEL_FEATURES].isnull().sum().sum())

# ── Build Fin-JEPA sequences: real customer_id this time, respecting split as-is ──
df_full = sequences.merge(label_timeline[['charge_id', 'train_label']], left_on='id', right_on='charge_id', how='left')
df_full = df_full.merge(ground_truth, left_on='id', right_on='charge_id', how='left', suffixes=('', '_gt'))
for col in MODEL_FEATURES:
    df_full[col] = df[col]  # bring over the cleaned/encoded columns

# Sliding-window builder for Fin-JEPA.
# Each window: [context_len] transactions as context, [pred_len] as target.
# Only customers with at least min_tx transactions are included.
# charge_id_list records the ID of the last target transaction (used for GT lookup).
def build_customer_windows(split_df, min_tx=3, context_len=4, pred_len=1, stride=1):
    ctx_list, tgt_list, id_list, charge_id_list = [], [], [], []
    grouped = split_df.sort_values('seq_pos').groupby('customer_id')
    for cid, g in grouped:
        if len(g) < min_tx:
            continue
        feats = g[MODEL_FEATURES].values.astype(np.float32)
        charge_ids = g['id'].values
        n = len(feats)
        total_len = context_len + pred_len
        if n < total_len:
            continue
        for start in range(0, n - total_len + 1, stride):
            ctx_list.append(feats[start: start + context_len])
            tgt_list.append(feats[start + context_len: start + total_len])
            id_list.append(cid)
            charge_id_list.append(charge_ids[start + total_len - 1])  # the charge being predicted
    if not ctx_list:
        return None, None, None, None
    return np.stack(ctx_list), np.stack(tgt_list), np.array(id_list), np.array(charge_id_list)

train_df_fj = df_full[df_full['split'] == 'train']
test_df_fj = df_full[df_full['split'] == 'test']

train_ctx, train_tgt, train_cust_ids, train_charge_ids = build_customer_windows(train_df_fj)
test_ctx, test_tgt, test_cust_ids, test_charge_ids = build_customer_windows(test_df_fj)

print(f"\nTrain windows: {train_ctx.shape}")
print(f"Test windows:  {test_ctx.shape}")

### Environment and object check

In [ ]:
import os
import numpy as np
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nExisting objects:")
for name in [
    "train_windows",
    "test_windows",
    "X_train",
    "X_test",
    "y_train",
    "y_test",
    "train_meta",
    "test_meta"
]:
    if name in globals():
        obj = globals()[name]
        try:
            print(f"{name}: shape={obj.shape}, type={type(obj).__name__}")
        except:
            print(f"{name}: type={type(obj).__name__}")

### Inspect ground truth and sequences

In [ ]:
import pandas as pd

DATA_DIR = "/kaggle/input/datasets/abhradwiplala/in-house-dataset"

gt = pd.read_csv(f"{DATA_DIR}/_ground_truth.csv")
seq = pd.read_csv(f"{DATA_DIR}/sequences.csv")

print("GROUND TRUTH")
print("Shape:", gt.shape)
print("Columns:", gt.columns.tolist())
print(gt.head())

print("\n" + "="*70)

print("SEQUENCES")
print("Shape:", seq.shape)
print("Columns:", seq.columns.tolist())
print(seq.head())

### Step 3: Verify charge mapping + official split + window logic

In [ ]:
import pandas as pd
import numpy as np

DATA_DIR = "/kaggle/input/datasets/abhradwiplala/in-house-dataset"

gt = pd.read_csv(f"{DATA_DIR}/_ground_truth.csv")
seq = pd.read_csv(f"{DATA_DIR}/sequences.csv")

print("=" * 70)
print("1. CHECK ID ↔ CHARGE_ID MAPPING")
print("=" * 70)

# Check whether every sequence id exists in ground truth
missing_gt = ~seq["id"].isin(gt["charge_id"])

print("Sequence rows:", len(seq))
print("Missing ground-truth IDs:", missing_gt.sum())

# Merge
merged = seq.merge(
    gt,
    left_on="id",
    right_on="charge_id",
    how="left",
    validate="one_to_one"
)

print("Merged shape:", merged.shape)
print("Missing is_fraud after merge:", merged["is_fraud"].isna().sum())

print("\nExample mapping:")
print(
    merged[
        ["id", "charge_id", "is_fraud", "typology", "split"]
    ].head(10)
)

In [ ]:
print("\n" + "=" * 70)
print("2. OFFICIAL TRAIN / TEST SPLIT")
print("=" * 70)

print(merged["split"].value_counts())

print("\nFraud rate by official split:")
print(
    merged.groupby("split")["is_fraud"]
    .mean()
)

In [ ]:
print("\n" + "=" * 70)
print("3. TYPOLOGY COUNTS")
print("=" * 70)

print(
    merged.groupby("typology")["is_fraud"]
    .agg(["count", "sum", "mean"])
    .sort_values("count", ascending=False)
)

In [ ]:
print("\n" + "=" * 70)
print("4. CUSTOMER SEQUENCE INFORMATION")
print("=" * 70)

# Sort exactly by customer and sequence position
seq_sorted = merged.sort_values(
    ["customer_id", "seq_pos"]
).reset_index(drop=True)

print("Number of customers:", seq_sorted["customer_id"].nunique())

print("\nSequence length statistics:")
customer_lengths = seq_sorted.groupby("customer_id").size()

print(customer_lengths.describe())

print("\nCustomers with >= 4 transactions:",
      (customer_lengths >= 4).sum())

In [ ]:
print("\n" + "=" * 70)
print("5. EXPECTED 4-STEP SLIDING WINDOWS")
print("=" * 70)

# For a 4-step sliding window:
# customer with N transactions contributes max(N-3, 0) windows

expected_windows = (customer_lengths - 3).clip(lower=0).sum()

print("Expected windows with window_size=4:",
      expected_windows)

print("\nPreviously observed:")
print("Train windows: 11670")
print("Test windows : 9632")

print("\nExpected train/test windows:")
for split_name in ["train", "test"]:
    split_lengths = (
        seq_sorted[seq_sorted["split"] == split_name]
        .groupby("customer_id")
        .size()
    )

    n_windows = (split_lengths - 3).clip(lower=0).sum()

    print(f"{split_name}: {n_windows}")

In [ ]:
print("\n" + "=" * 70)
print("6. CHECK SEQUENCE POSITION")
print("=" * 70)

print(
    seq_sorted[
        [
            "customer_id",
            "seq_pos",
            "id",
            "created",
            "split",
            "is_fraud",
            "typology"
        ]
    ].head(20).to_string(index=False)
)

### Step 4A: Check what preprocessing variables currently exist

In [ ]:
import numpy as np
import pandas as pd

print("=" * 70)
print("CURRENT NOTEBOOK VARIABLES")
print("=" * 70)

for name, obj in list(globals().items()):
    if name.startswith("_"):
        continue

    try:
        if hasattr(obj, "shape"):
            print(f"{name:35s} shape={obj.shape} type={type(obj).__name__}")
    except:
        pass

print("\n" + "=" * 70)
print("LOOKING FOR PREPROCESSING-RELATED VARIABLES")
print("=" * 70)

keywords = [
    "merged",
    "train",
    "test",
    "feature",
    "numeric",
    "categor",
    "window",
    "sequence",
    "scaler",
    "encoder",
    "X_"
]

for name in globals():
    if any(k in name.lower() for k in keywords):
        print(name)

### Step 5: Verify Fin-JEPA Target ↔ Ground Truth Alignment

In [ ]:
import numpy as np
import pandas as pd

print("=" * 70)
print("WINDOW / TARGET ALIGNMENT")
print("=" * 70)

print("Train context :", train_ctx.shape)
print("Train target  :", train_tgt.shape)
print("Train charges :", train_charge_ids.shape)

print("\nTest context  :", test_ctx.shape)
print("Test target   :", test_tgt.shape)
print("Test charges  :", test_charge_ids.shape)

#### Build ground-truth lookup

In [ ]:
gt_lookup = (
    ground_truth
    .set_index("charge_id")[["is_fraud", "typology"]]
)

#### Map every test target to its ground truth

In [ ]:
test_labels = gt_lookup.loc[test_charge_ids].reset_index()

test_labels.columns = ["charge_id", "is_fraud", "typology"]

#### Verify no missing labels

In [ ]:
print("\n" + "=" * 70)
print("GROUND TRUTH ALIGNMENT")
print("=" * 70)

print("Test targets:", len(test_labels))
print("Missing fraud labels:",
      test_labels["is_fraud"].isna().sum())
print("Missing typology:",
      test_labels["typology"].isna().sum())

#### Compare test target distribution

In [ ]:
print("\n" + "=" * 70)
print("TEST TARGET DISTRIBUTION")
print("=" * 70)

print(
    test_labels["is_fraud"]
    .value_counts()
    .rename(index={False: "Normal", True: "Fraud"})
)

print("\nFraud rate:",
      test_labels["is_fraud"].mean())

In [ ]:
print("\n" + "=" * 70)
print("TYPOLOGY DISTRIBUTION")
print("=" * 70)

print(
    test_labels.groupby("typology")["is_fraud"]
    .agg(["count", "sum", "mean"])
    .sort_values("count", ascending=False)
)

#### Check duplicate target charge IDs

In [ ]:
print("\n" + "=" * 70)
print("TARGET ID UNIQUENESS")
print("=" * 70)

print("Total target IDs:", len(test_charge_ids))
print("Unique target IDs:", len(np.unique(test_charge_ids)))
print("Duplicate target IDs:",
      len(test_charge_ids) - len(np.unique(test_charge_ids)))

#### Show examples

In [ ]:
print("\n" + "=" * 70)
print("FIRST 20 TEST TARGETS")
print("=" * 70)

print(test_labels.head(20).to_string(index=False))

## Part 2: Fin-JEPA training and evaluation

Train the Fin-JEPA sequence model on customer windows, then score the test set and evaluate it overall, at fixed k, at fixed recall levels and by fraud typology.

### Step 6: Fin-JEPA Training

- In-house dataset

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

#### 1. Dataset

In [ ]:
# Dataset wrapper for Fin-JEPA.
# ctx: (N, context_len, n_features)  - context transaction sequences
# tgt: (N, pred_len, n_features)     - target transactions to predict
class JEPAWindowDataset(Dataset):

    def __init__(self, ctx, tgt):
        self.ctx = torch.tensor(ctx, dtype=torch.float32)
        self.tgt = torch.tensor(tgt, dtype=torch.float32)

    def __len__(self):
        return len(self.ctx)

    def __getitem__(self, idx):
        return self.ctx[idx], self.tgt[idx]

In [ ]:
train_dataset = JEPAWindowDataset(
    train_ctx,
    train_tgt
)

test_dataset = JEPAWindowDataset(
    test_ctx,
    test_tgt
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train samples:", len(train_dataset))
print("Test samples :", len(test_dataset))

#### 2. Fin-JEPA Encoder

In [ ]:
# Per-transaction feature encoder for Fin-JEPA.
# Projects raw features -> embed_dim via an MLP + LayerNorm.
# Applied independently to each transaction in a sequence.
class PriceEncoder(nn.Module):

    def __init__(
        self,
        n_features=35,
        embed_dim=64,
        hidden_dim=128
    ):
        super().__init__()

        self.input_proj = nn.Linear(
            n_features,
            embed_dim
        )

        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),

            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),

            nn.Linear(hidden_dim, embed_dim)
        )

        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):

        x = self.input_proj(x)
        x = self.mlp(x)
        x = self.norm(x)

        return x

#### 3. Causal Transformer Predictor

In [ ]:
# Causal Transformer that reads context embeddings and predicts the
# next transaction's latent representation.
# Upper-triangular mask enforces causality: position i cannot attend to j > i.
class TransformerPredictor(nn.Module):

    def __init__(
        self,
        embed_dim=64,
        n_heads=4,
        n_layers=4,
        dropout=0.1
    ):
        super().__init__()

        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            layer,
            num_layers=n_layers
        )

        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):

        seq_len = x.size(1)

        # Causal mask:
        # position i cannot see future positions
        mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len,
                device=x.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        x = self.transformer(
            x,
            mask=mask
        )

        return self.norm(x)

#### 4. SIGReg

In [ ]:
# Spectral Information Geometry Regulariser.
# Projects embeddings onto random directions and penalises deviation from
# a standard-normal distribution. Prevents embedding collapse during
# self-supervised training without negative samples.
class SIGReg(nn.Module):

    def __init__(
        self,
        knots=17,
        num_proj=128
    ):
        super().__init__()

        self.knots = knots
        self.num_proj = num_proj

    def forward(self, z):

        # Flatten all temporal positions
        z = z.reshape(-1, z.shape[-1])

        # Normalize dimensions
        z = (z - z.mean(dim=0, keepdim=True)) / (
            z.std(dim=0, keepdim=True) + 1e-6
        )

        # Random projections
        projection = torch.randn(
            z.shape[-1],
            self.num_proj,
            device=z.device
        )

        projection = F.normalize(
            projection,
            dim=0
        )

        projected = z @ projection

        # Encourage approximately Gaussian projected
        # distributions.
        sorted_vals, _ = torch.sort(
            projected,
            dim=0
        )

        n = sorted_vals.shape[0]

        # Standard normal quantiles
        probs = (
            torch.arange(
                1,
                n + 1,
                device=z.device,
                dtype=torch.float32
            ) - 0.5
        ) / n

        normal_quantiles = (
            torch.erfinv(
                2 * probs - 1
            ) * math.sqrt(2.0)
        )

        normal_quantiles = normal_quantiles.unsqueeze(1)

        loss = F.mse_loss(
            sorted_vals,
            normal_quantiles.expand_as(sorted_vals)
        )

        return loss

#### 5. Fin-JEPA Model

In [ ]:
# Full Fin-JEPA model.
# Architecture: online encoder + causal predictor predict the target encoder's
# representation of the next transaction (JEPA objective).
# The target encoder is frozen during forward; it is updated via EMA externally.
# Loss = MSE(predicted latent, target latent) + 0.1 * SIGReg.
class FinJEPA(nn.Module):

    def __init__(
        self,
        n_features=35,
        embed_dim=64
    ):
        super().__init__()

        self.encoder = PriceEncoder(
            n_features=n_features,
            embed_dim=embed_dim
        )

        self.predictor = TransformerPredictor(
            embed_dim=embed_dim,
            n_heads=4,
            n_layers=4
        )

        self.target_encoder = PriceEncoder(
            n_features=n_features,
            embed_dim=embed_dim
        )

        self.sigreg = SIGReg()

    def forward(self, context, target):

        # ----------------------------------------------------
        # Encode context
        # ----------------------------------------------------

        context_embeddings = self.encoder(
            context
        )

        # ----------------------------------------------------
        # Predict latent representations
        # ----------------------------------------------------

        predicted = self.predictor(
            context_embeddings
        )

        # Use final context representation to predict target
        predicted_target = predicted[:, -1:, :]

        # ----------------------------------------------------
        # Encode target
        # ----------------------------------------------------

        with torch.no_grad():

            target_embeddings = self.target_encoder(
                target
            )

        # ----------------------------------------------------
        # Prediction loss
        # ----------------------------------------------------

        pred_loss = F.mse_loss(
            predicted_target,
            target_embeddings
        )

        # ----------------------------------------------------
        # SIGReg
        # ----------------------------------------------------

        combined = torch.cat(
            [
                context_embeddings,
                target_embeddings
            ],
            dim=1
        )

        sigreg_loss = self.sigreg(
            combined
        )

        # ----------------------------------------------------
        # Total loss
        # ----------------------------------------------------

        total_loss = (
            pred_loss
            + 0.1 * sigreg_loss
        )

        return {
            "loss": total_loss,
            "pred_loss": pred_loss,
            "sigreg_loss": sigreg_loss,
            "predicted": predicted_target,
            "target": target_embeddings
        }

#### 6. Create Model

In [ ]:
model = FinJEPA(
    n_features=35,
    embed_dim=64
).to(device)

print("\nModel created.")

num_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Trainable parameters:", f"{num_params:,}")

#### 7. Optimizer

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=20
)

#### 8. Training

In [ ]:
EPOCHS = 20

history = []

print("\n" + "=" * 70)
print("STARTING FIN-JEPA TRAINING")
print("=" * 70)

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0.0
    total_pred_loss = 0.0
    total_sigreg_loss = 0.0

    for ctx, tgt in train_loader:

        ctx = ctx.to(
            device,
            non_blocking=True
        )

        tgt = tgt.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        output = model(
            ctx,
            tgt
        )

        loss = output["loss"]

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()
        total_pred_loss += output["pred_loss"].item()
        total_sigreg_loss += output["sigreg_loss"].item()

    scheduler.step()

    avg_loss = (
        total_loss / len(train_loader)
    )

    avg_pred = (
        total_pred_loss / len(train_loader)
    )

    avg_sigreg = (
        total_sigreg_loss / len(train_loader)
    )

    history.append(
        [
            epoch + 1,
            avg_loss,
            avg_pred,
            avg_sigreg,
            scheduler.get_last_lr()[0]
        ]
    )

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Loss: {avg_loss:.6f} | "
        f"Pred: {avg_pred:.6f} | "
        f"SIGReg: {avg_sigreg:.6f} | "
        f"LR: {scheduler.get_last_lr()[0]:.2e}"
    )

print("\nTraining complete.")

### Step 7: Fin-JEPA Test-Time Anomaly Scores

In [ ]:
# Score every test window: anomaly score = mean squared latent prediction error.
# Higher score -> the model found the next transaction harder to predict,
# which is the unsupervised fraud signal.
model.eval()

all_scores = []
all_pred_losses = []

with torch.no_grad():

    for ctx, tgt in test_loader:

        ctx = ctx.to(
            device,
            non_blocking=True
        )

        tgt = tgt.to(
            device,
            non_blocking=True
        )

        output = model(
            ctx,
            tgt
        )

        predicted = output["predicted"]
        target = output["target"]

        # ----------------------------------------------------
        # Per-sample latent prediction error
        # ----------------------------------------------------

        sample_error = (
            (predicted - target) ** 2
        ).mean(dim=(1, 2))

        all_scores.append(
            sample_error.detach()
            .cpu()
            .numpy()
        )

        all_pred_losses.append(
            output["pred_loss"]
            .detach()
            .cpu()
            .numpy()
        )

In [ ]:
# Combine batches
test_scores = np.concatenate(
    all_scores
)

test_pred_losses = np.array(
    all_pred_losses
)

print("=" * 70)
print("FIN-JEPA TEST SCORING")
print("=" * 70)

print("Number of scores:", len(test_scores))

print("\nScore statistics:")
print(
    pd.Series(test_scores).describe()
)

print("\nNaN scores:", np.isnan(test_scores).sum())
print("Inf scores:", np.isinf(test_scores).sum())

#### Attach scores to ground-truth targets

In [ ]:
results = test_labels.copy()

results["fin_jepa_score"] = test_scores

print("\n" + "=" * 70)
print("SCORED TEST DATA")
print("=" * 70)

print(results.head(20).to_string(index=False))

print("\nScore by fraud status:")
print(
    results.groupby("is_fraud")["fin_jepa_score"]
    .agg(["count", "mean", "median", "std", "min", "max"])
)

In [ ]:
print("\nScore by typology:")
print(
    results.groupby("typology")["fin_jepa_score"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .sort_values("mean", ascending=False)
)

### Step 8: Overall Fin-JEPA Evaluation

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

y_true = results["is_fraud"].astype(int).values
y_score = results["fin_jepa_score"].values

#### 1. ROC-AUC

In [ ]:
roc_auc = roc_auc_score(
    y_true,
    y_score
)

#### 2. PR-AUC

In [ ]:
pr_auc = average_precision_score(
    y_true,
    y_score
)

#### 3. Threshold-based metrics

In [ ]:
# Use the median score as a threshold so ~50% of transactions are flagged;
# this is informative but not how the model would be deployed in practice.
threshold = np.median(y_score)

y_pred = (
    y_score >= threshold
).astype(int)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

cm = confusion_matrix(
    y_true,
    y_pred
)

#### Print

In [ ]:
print("=" * 70)
print("FIN-JEPA OVERALL TEST RESULTS")
print("=" * 70)

print(f"Evaluation samples : {len(y_true)}")
print(f"Fraud samples      : {y_true.sum()}")
print(f"Normal samples     : {(y_true == 0).sum()}")

print(f"\nROC-AUC : {roc_auc:.4f}")
print(f"PR-AUC  : {pr_auc:.4f}")

print("\nThreshold-based metrics")
print(f"Threshold : {threshold:.6f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")

print("\nConfusion matrix")
print(cm)

### Step 9: Precision@k / Recall@k

In [ ]:
# Compute Precision@K and Recall@K for a given top-fraction of transactions.
# fraction=0.01 means 'flag the 1% of test transactions with the highest anomaly score'.
def precision_recall_at_k(y_true, scores, fraction):

    n = len(y_true)

    k = max(
        1,
        int(np.ceil(n * fraction))
    )

    order = np.argsort(scores)[::-1]
    top_indices = order[:k]

    selected = y_true[top_indices]

    precision_k = selected.mean()

    total_positives = y_true.sum()

    recall_k = (
        selected.sum() / total_positives
        if total_positives > 0
        else 0.0
    )

    return (
        k,
        precision_k,
        recall_k,
        selected.sum()
    )

In [ ]:
k_levels = [
    0.001,
    0.005,
    0.01,
    0.02,
    0.05,
    0.10
]

rows = []

for fraction in k_levels:

    k, precision_k, recall_k, fraud_found = (
        precision_recall_at_k(
            y_true,
            y_score,
            fraction
        )
    )

    rows.append({
        "Top_K": f"{fraction * 100:g}%",
        "K_samples": k,
        "Frauds_found": fraud_found,
        "Precision@K": precision_k,
        "Recall@K": recall_k
    })

In [ ]:
prk_table = pd.DataFrame(rows)

print("=" * 70)
print("FIN-JEPA PRECISION@K / RECALL@K")
print("=" * 70)

print(
    prk_table.to_string(
        index=False,
        formatters={
            "Precision@K": "{:.4f}".format,
            "Recall@K": "{:.4f}".format
        }
    )
)

### Step 10: Typology-Split Fin-JEPA Evaluation

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

print("=" * 70)
print("FIN-JEPA TYPOLOGY-SPLIT EVALUATION")
print("=" * 70)

typology_results = []

for typology in sorted(results["typology"].unique()):

    subset = results[
        results["typology"] == typology
    ].copy()

    y = subset["is_fraud"].astype(int).values
    scores = subset["fin_jepa_score"].values

    n = len(subset)
    frauds = y.sum()
    normals = n - frauds

    print("\n" + "-" * 70)
    print(f"TYPOLOGY: {typology}")
    print("-" * 70)

    print("Samples :", n)
    print("Frauds  :", frauds)
    print("Normal  :", normals)

    # --------------------------------------------------------
    # ROC-AUC / PR-AUC require both classes
    # --------------------------------------------------------

    if len(np.unique(y)) == 2:

        roc = roc_auc_score(
            y,
            scores
        )

        pr = average_precision_score(
            y,
            scores
        )

        print(f"ROC-AUC : {roc:.4f}")
        print(f"PR-AUC  : {pr:.4f}")

        typology_results.append({
            "typology": typology,
            "samples": n,
            "frauds": frauds,
            "normal": normals,
            "ROC-AUC": roc,
            "PR-AUC": pr
        })

    else:

        print(
            "ROC-AUC / PR-AUC: not defined "
            "(only one class present)"
        )

        typology_results.append({
            "typology": typology,
            "samples": n,
            "frauds": frauds,
            "normal": normals,
            "ROC-AUC": np.nan,
            "PR-AUC": np.nan
        })

In [ ]:
typology_table = pd.DataFrame(
    typology_results
)

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

print(
    typology_table.to_string(
        index=False,
        formatters={
            "ROC-AUC": lambda x:
                "N/A" if pd.isna(x) else f"{x:.4f}",
            "PR-AUC": lambda x:
                "N/A" if pd.isna(x) else f"{x:.4f}"
        }
    )
)

### Step 11: Precision at Fixed Recall Levels

In [ ]:
from sklearn.metrics import precision_recall_curve

precision_curve, recall_curve, thresholds = precision_recall_curve(
    y_true,
    y_score
)

target_recalls = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]

rows = []

for target_recall in target_recalls:

    # Find the point whose recall is closest to requested recall
    idx = np.argmin(
        np.abs(
            recall_curve - target_recall
        )
    )

    rows.append({
        "Target Recall": target_recall,
        "Actual Recall": recall_curve[idx],
        "Precision": precision_curve[idx]
    })

In [ ]:
recall_precision_table = pd.DataFrame(rows)

print("=" * 70)
print("FIN-JEPA PRECISION AT FIXED RECALL LEVELS")
print("=" * 70)

print(
    recall_precision_table.to_string(
        index=False,
        formatters={
            "Target Recall": "{:.0%}".format,
            "Actual Recall": "{:.4f}".format,
            "Precision": "{:.4f}".format
        }
    )
)

## Part 3: T-JEPA training and evaluation

Prepare tabular features, define the feature-masking T-JEPA model, pretrain it, and run the same evaluations as for Fin-JEPA.

### Step 12: T-JEPA Feature Preparation

In [ ]:
print("=" * 70)
print("T-JEPA INPUT FEATURES")
print("=" * 70)

print("Number of MODEL_FEATURES:", len(MODEL_FEATURES))

for i, feature in enumerate(MODEL_FEATURES):
    print(f"{i:02d}: {feature}")

In [ ]:
print("\n" + "=" * 70)
print("FEATURE MATRIX")
print("=" * 70)

X_train_tj = train_df_fj[MODEL_FEATURES].copy()
X_test_tj = test_df_fj[MODEL_FEATURES].copy()

print("X_train shape:", X_train_tj.shape)
print("X_test shape :", X_test_tj.shape)

print("\nNaNs in train:", X_train_tj.isna().sum().sum())
print("NaNs in test :", X_test_tj.isna().sum().sum())

In [ ]:
print("\n" + "=" * 70)
print("DATA TYPES")
print("=" * 70)

print(X_train_tj.dtypes.value_counts())

In [ ]:
print("\n" + "=" * 70)
print("FIRST 5 ROWS")
print("=" * 70)

print(X_train_tj.head())

### Step 13: T-JEPA Model + Feature Masking

In [ ]:
import copy
import math
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

#### 1. Reproducibility

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

#### 2. Device

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

#### 3. Convert dataframe -> numpy

In [ ]:
X_train_np = X_train_tj.values.astype(
    np.float32
)

X_test_np = X_test_tj.values.astype(
    np.float32
)

n_features = X_train_np.shape[1]

print("\nTrain:", X_train_np.shape)
print("Test :", X_test_np.shape)
print("Features:", n_features)

#### 4. Dataset

In [ ]:
# Minimal Dataset wrapper for flat tabular rows (T-JEPA input).
# Returns a single feature vector per index; masking is applied in the training loop.
class TabularDataset(Dataset):

    def __init__(self, X):
        self.X = torch.from_numpy(X)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx]

In [ ]:
train_dataset_tj = TabularDataset(
    X_train_np
)

test_dataset_tj = TabularDataset(
    X_test_np
)

#### 5. DataLoader

In [ ]:
BATCH_SIZE = 256

train_loader_tj = DataLoader(
    train_dataset_tj,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    drop_last=True
)

test_loader_tj = DataLoader(
    test_dataset_tj,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

#### 6. Feature masking

- Context: 75-85% of features
- Target : 15-25% of features
- Context and target are disjoint.

In [ ]:
# Generates per-sample random context/target feature masks for T-JEPA.
# Context and target partitions are disjoint; a random permutation ensures this.
# Mask sizes are sampled uniformly within [min_*, max_*] feature counts each call.
class FeatureMasker:

    def __init__(
        self,
        n_features,
        min_context_share=0.75,
        max_context_share=0.85,
        min_target_share=0.15,
        max_target_share=0.25
    ):

        self.n_features = n_features

        self.min_context = max(
            1,
            int(
                math.ceil(
                    n_features * min_context_share
                )
            )
        )

        self.max_context = min(
            n_features - 1,
            int(
                math.floor(
                    n_features * max_context_share
                )
            )
        )

        self.min_target = max(
            1,
            int(
                math.ceil(
                    n_features * min_target_share
                )
            )
        )

        self.max_target = min(
            n_features - 1,
            int(
                math.floor(
                    n_features * max_target_share
                )
            )
        )


    def __call__(self, batch_size, device):

        context_masks = []
        target_masks = []

        for _ in range(batch_size):

            # Random target size
            target_size = random.randint(
                self.min_target,
                self.max_target
            )

            # Random context size
            context_size = random.randint(
                self.min_context,
                self.max_context
            )

            permutation = torch.randperm(
                self.n_features
            )

            target_idx = permutation[
                :target_size
            ]

            remaining = permutation[
                target_size:
            ]

            # Make sure context does not overlap target
            context_idx = remaining[
                :context_size
            ]

            context_mask = torch.zeros(
                self.n_features,
                dtype=torch.bool
            )

            target_mask = torch.zeros(
                self.n_features,
                dtype=torch.bool
            )

            context_mask[
                context_idx
            ] = True

            target_mask[
                target_idx
            ] = True

            context_masks.append(
                context_mask
            )

            target_masks.append(
                target_mask
            )

        return (
            torch.stack(context_masks).to(device),
            torch.stack(target_masks).to(device)
        )

In [ ]:
masker = FeatureMasker(
    n_features=n_features,
    min_context_share=0.75,
    max_context_share=0.85,
    min_target_share=0.15,
    max_target_share=0.25
)

print("\nMasking configuration:")
print(
    "Context features:",
    masker.min_context,
    "-",
    masker.max_context
)

print(
    "Target features :",
    masker.min_target,
    "-",
    masker.max_target
)

#### 7. Feature tokenizer

- Each scalar feature becomes a feature token:
- scalar value + feature identity -> embedding

In [ ]:
# Converts a flat feature vector into a sequence of per-feature tokens.
# Each token = linear projection of the scalar value + learned feature-position embedding.
# This lets the Transformer treat features as a set, not a flat concatenation.
class FeatureTokenizer(nn.Module):

    def __init__(
        self,
        n_features,
        embed_dim
    ):

        super().__init__()

        self.value_projection = nn.Linear(
            1,
            embed_dim
        )

        self.feature_embedding = nn.Parameter(
            torch.randn(
                n_features,
                embed_dim
            ) * 0.02
        )

        self.norm = nn.LayerNorm(
            embed_dim
        )


    def forward(self, x):

        # x: [B, F]

        value_tokens = self.value_projection(
            x.unsqueeze(-1)
        )

        feature_tokens = (
            self.feature_embedding
            .unsqueeze(0)
        )

        tokens = (
            value_tokens
            + feature_tokens
        )

        return self.norm(tokens)

#### 8. Context encoder

In [ ]:
# Encodes the visible (context) features using a Transformer.
# Non-context feature tokens are zeroed out before the Transformer so the
# model only attends to the features it is allowed to see.
class ContextEncoder(nn.Module):

    def __init__(
        self,
        n_features,
        embed_dim=128,
        n_heads=4,
        n_layers=3,
        dropout=0.1
    ):

        super().__init__()

        self.tokenizer = FeatureTokenizer(
            n_features,
            embed_dim
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation="gelu"
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers
        )

        self.norm = nn.LayerNorm(
            embed_dim
        )


    def forward(
        self,
        x,
        context_mask
    ):

        tokens = self.tokenizer(x)

        # Mask features not belonging to context
        visible = context_mask.unsqueeze(-1)

        tokens = tokens * visible

        encoded = self.transformer(
            tokens
        )

        return self.norm(encoded)

#### 9. Target encoder

- Momentum/EMA copy of context encoder.

In [ ]:
# EMA (momentum) copy of the context encoder used to produce stable target representations.
# All parameters are frozen (requires_grad=False); weights are updated via
# exponential moving average of the context encoder in the training loop.
class TargetEncoder(nn.Module):

    def __init__(
        self,
        context_encoder
    ):

        super().__init__()

        self.encoder = copy.deepcopy(
            context_encoder
        )

        for parameter in self.encoder.parameters():
            parameter.requires_grad = False


    @torch.no_grad()
    def forward(
        self,
        x,
        target_mask
    ):

        tokens = self.encoder.tokenizer(x)

        encoded = self.encoder.transformer(
            tokens
        )

        encoded = self.encoder.norm(
            encoded
        )

        return encoded

#### 10. Predictor

- Takes context representation + target feature identity
- and predicts the target latent representation.

In [ ]:
# Predicts target feature representations from the context encoding.
# NOTE: this original version has a dimension bug that is fixed in Step 14A.
class TargetPredictor(nn.Module):

    def __init__(
        self,
        embed_dim=128,
        n_heads=4,
        n_layers=2,
        dropout=0.1
    ):

        super().__init__()

        decoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation="gelu"
        )

        self.transformer = nn.TransformerEncoder(
            decoder_layer,
            num_layers=n_layers
        )

        self.norm = nn.LayerNorm(
            embed_dim
        )


        self.output_projection = nn.Linear(
            embed_dim,
            embed_dim
        )


    def forward(
        self,
        context_tokens,
        target_mask,
        feature_embedding
    ):

        B, F, D = context_tokens.shape

        # Global context representation
        context_summary = (
            context_tokens
            .sum(dim=1)
            /
            context_tokens.abs()
            .sum(dim=1)
            .clamp(min=1.0)
            .unsqueeze(-1)
        )

        context_summary = (
            context_summary
            .unsqueeze(1)
            .expand(-1, F, -1)
        )

        # Target feature identity
        target_tokens = (
            feature_embedding
            .unsqueeze(0)
            .expand(B, -1, -1)
        )

        predictor_input = (
            context_summary
            + target_tokens
        )

        predicted = self.transformer(
            predictor_input
        )

        predicted = self.norm(
            predicted
        )

        predicted = self.output_projection(
            predicted
        )

        return predicted

#### 11. Complete T-JEPA

In [ ]:
# Complete T-JEPA model.
# JEPA objective: context encoder + predictor jointly predict the EMA target
# encoder's token-level representations for masked (target) features.
# Only context_encoder and predictor have trainable parameters.
class TJEPA(nn.Module):

    def __init__(
        self,
        n_features,
        embed_dim=128,
        n_heads=4,
        encoder_layers=3,
        predictor_layers=2,
        dropout=0.1,
        ema_decay=0.996
    ):

        super().__init__()

        self.context_encoder = ContextEncoder(
            n_features=n_features,
            embed_dim=embed_dim,
            n_heads=n_heads,
            n_layers=encoder_layers,
            dropout=dropout
        )

        self.target_encoder = TargetEncoder(
            self.context_encoder
        )

        self.predictor = TargetPredictor(
            embed_dim=embed_dim,
            n_heads=n_heads,
            n_layers=predictor_layers,
            dropout=dropout
        )

        self.ema_decay = ema_decay


    # Polyak EMA update: θ_target ← τ·θ_target + (1-τ)·θ_context
    # Called once per batch; τ (ema_decay) increases from 0.996 → 0.999 over training.
    @torch.no_grad()
    def update_target_encoder(self):

        for target_param, context_param in zip(
            self.target_encoder.parameters(),
            self.context_encoder.parameters()
        ):

            target_param.data.mul_(
                self.ema_decay
            )

            target_param.data.add_(
                context_param.data,
                alpha=1.0 - self.ema_decay
            )


    def forward(
        self,
        x,
        context_mask,
        target_mask
    ):

        # ----------------------------------------------------
        # Context representation
        # ----------------------------------------------------

        context_tokens = self.context_encoder(
            x,
            context_mask
        )

        # ----------------------------------------------------
        # Target representation
        # ----------------------------------------------------

        with torch.no_grad():

            target_tokens = self.target_encoder(
                x,
                target_mask
            )


        # ----------------------------------------------------
        # Predictions
        # ----------------------------------------------------

        predictions = self.predictor(
            context_tokens,
            target_mask,
            self.context_encoder.tokenizer.feature_embedding
        )

        return (
            predictions,
            target_tokens
        )

#### 12. Create model

In [ ]:
tjepa = TJEPA(
    n_features=n_features,
    embed_dim=128,
    n_heads=4,
    encoder_layers=3,
    predictor_layers=2,
    dropout=0.1,
    ema_decay=0.996
).to(device)

#### 13. Count parameters

In [ ]:
trainable_params = sum(
    p.numel()
    for p in tjepa.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in tjepa.parameters()
)

print("\n" + "=" * 70)
print("T-JEPA MODEL")
print("=" * 70)

print(
    "Trainable parameters:",
    f"{trainable_params:,}"
)

print(
    "Total parameters:",
    f"{total_params:,}"
)

#### 14. Test masking once

In [ ]:
sample_batch = next(
    iter(train_loader_tj)
).to(device)

context_mask, target_mask = masker(
    sample_batch.shape[0],
    device
)

print("\nMask sanity check:")

print(
    "Context features/sample:",
    context_mask[0].sum().item()
)

print(
    "Target features/sample:",
    target_mask[0].sum().item()
)

print(
    "Overlap:",
    (
        context_mask
        & target_mask
    )[0].sum().item()
)

print(
    "Unmasked:",
    (
        ~(
            context_mask
            | target_mask
        )
    )[0].sum().item()
)

print("\nModel ready.")

### Step 14A: Fix T-JEPA Target Predictor

In [ ]:
# Corrected TargetPredictor (Step 14A).
# Key fix: context summary is computed as an average over available (non-zero)
# context tokens rather than a weighted sum with unstable normalization.
# Own feature_embedding is independent from the tokenizer's embedding.
class TargetPredictor(nn.Module):

    def __init__(
        self,
        embed_dim,
        num_heads,
        num_layers,
        dropout,
        num_features
    ):
        super().__init__()

        self.embed_dim = embed_dim
        self.num_features = num_features

        # Learned embedding for each feature position
        self.feature_embedding = nn.Parameter(
            torch.randn(
                1,
                num_features,
                embed_dim
            ) * 0.02
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation="gelu"
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.output_norm = nn.LayerNorm(embed_dim)

        self.output_projection = nn.Linear(
            embed_dim,
            embed_dim
        )


    def forward(
        self,
        context_tokens,
        target_mask,
        feature_embedding
    ):
        """
        context_tokens:
            [B, F, D]

        target_mask:
            [B, F]

        feature_embedding:
            [1, F, D]
        """

        batch_size = context_tokens.shape[0]

        # ----------------------------------------------------
        # Context summary
        # ----------------------------------------------------
        #
        # context_tokens are already zeroed for non-context
        # features by the context encoder.
        #
        # Sum only the available context representations.
        # ----------------------------------------------------

        context_available = (
            context_tokens.abs().sum(dim=-1) > 0
        ).float()

        context_count = (
            context_available.sum(
                dim=1,
                keepdim=True
            ).clamp(min=1.0)
        )

        context_summary = (
            context_tokens.sum(dim=1)
            /
            context_count
        )

        # [B, D]
        context_summary = context_summary.unsqueeze(1)

        # [B, 1, D]

        # ----------------------------------------------------
        # Expand context summary to every feature
        # ----------------------------------------------------

        context_summary = context_summary.expand(
            batch_size,
            self.num_features,
            self.embed_dim
        )

        # ----------------------------------------------------
        # Add feature identity
        # ----------------------------------------------------

        feature_tokens = (
            context_summary
            +
            self.feature_embedding
        )

        # ----------------------------------------------------
        # Transformer predictor
        # ----------------------------------------------------

        predicted = self.transformer(
            feature_tokens
        )

        predicted = self.output_norm(
            predicted
        )

        predicted = self.output_projection(
            predicted
        )

        return predicted

### Step 14B: Inspect Existing T-JEPA

In [ ]:
import inspect

print("=" * 70)
print("TJEPA CONSTRUCTOR")
print("=" * 70)

print(inspect.signature(TJEPA))

print("\n" + "=" * 70)
print("CURRENT TJEPA COMPONENTS")
print("=" * 70)

print("tjepa type:", type(tjepa))

print("\nPredictor type:")
print(type(tjepa.predictor))

print("\nPredictor constructor:")
print(inspect.signature(TargetPredictor))

print("\nPredictor:")
print(tjepa.predictor)

### Step 14C: Replace Broken Predictor

In [ ]:
tjepa.predictor = TargetPredictor(
    embed_dim=128,
    num_heads=4,
    num_layers=2,
    dropout=0.1,
    num_features=35
).to(device)

print("=" * 70)
print("PREDICTOR REPLACED")
print("=" * 70)

print(tjepa.predictor)

### Step 14D: Forward Pass Test

In [ ]:
tjepa.eval()

with torch.no_grad():

    test_batch = next(
        iter(train_loader_tj)
    ).to(device)

    context_mask, target_mask = masker(
        test_batch.shape[0],
        device
    )

    predictions, targets = tjepa(
        test_batch,
        context_mask,
        target_mask
    )

print("=" * 70)
print("FORWARD PASS TEST")
print("=" * 70)

print("Input shape     :", test_batch.shape)
print("Context mask    :", context_mask.shape)
print("Target mask     :", target_mask.shape)
print("Predictions     :", predictions.shape)
print("Targets         :", targets.shape)

print(
    "Prediction NaN  :",
    torch.isnan(predictions).any().item()
)

print(
    "Target NaN      :",
    torch.isnan(targets).any().item()
)

print(
    "Prediction Inf  :",
    torch.isinf(predictions).any().item()
)

print(
    "Target Inf      :",
    torch.isinf(targets).any().item()
)

### Step 15: T-JEPA Self-Supervised Pretraining

In [ ]:
import time
import copy

# ------------------------------------------------------------
# Hyperparameters
# ------------------------------------------------------------

# T-JEPA pretraining hyperparameters.
# Only context_encoder and predictor parameters are optimised;
# target_encoder is updated via EMA and excluded from the optimiser.
TJ_EPOCHS = 30
TJ_LR = 1e-4
TJ_WEIGHT_DECAY = 1e-5

TJ_EMA_START = 0.996
TJ_EMA_FINAL = 0.999

# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer_tj = torch.optim.AdamW(
    [
        {
            "params": tjepa.context_encoder.parameters()
        },
        {
            "params": tjepa.predictor.parameters()
        }
    ],
    lr=TJ_LR,
    weight_decay=TJ_WEIGHT_DECAY
)

# ------------------------------------------------------------
# Scheduler
# ------------------------------------------------------------

scheduler_tj = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_tj,
    T_max=TJ_EPOCHS,
    eta_min=1e-6
)

# ------------------------------------------------------------
# JEPA loss
# ------------------------------------------------------------

# Masked cosine similarity loss: minimise 1 - cos(pred, target) for target features only.
# Cosine loss is more stable than MSE for normalised latent vectors.
def tjepa_loss(
    predictions,
    targets,
    target_mask
):

    prediction_norm = F.normalize(
        predictions,
        dim=-1
    )

    target_norm = F.normalize(
        targets,
        dim=-1
    )

    cosine_loss = 1.0 - (
        prediction_norm * target_norm
    ).sum(dim=-1)

    masked_loss = cosine_loss[target_mask]

    return masked_loss.mean()

#### EMA schedule

In [ ]:
# Linearly increase EMA decay from start to final over training.
# Higher decay late in training -> more stable target representations.
def get_ema_decay(
    epoch,
    total_epochs,
    start=0.996,
    final=0.999
):

    progress = epoch / max(
        1,
        total_epochs - 1
    )

    return (
        start
        +
        (final - start) * progress
    )

#### Training

In [ ]:
print("=" * 70)
print("STARTING T-JEPA PRETRAINING")
print("=" * 70)

print("Epochs        :", TJ_EPOCHS)
print("Learning rate :", TJ_LR)
print("Weight decay  :", TJ_WEIGHT_DECAY)
print("Batch size    :", BATCH_SIZE)

training_history_tj = []

best_loss = float("inf")
best_state_tj = None

start_time = time.time()

In [ ]:
for epoch in range(TJ_EPOCHS):

    tjepa.train()

    epoch_loss = 0.0
    batches = 0

    current_ema = get_ema_decay(
        epoch,
        TJ_EPOCHS,
        TJ_EMA_START,
        TJ_EMA_FINAL
    )

    tjepa.ema_decay = current_ema


    for batch in train_loader_tj:

        batch = batch.to(
            device,
            non_blocking=True
        )

        # ----------------------------------------------------
        # Fresh random masking
        # ----------------------------------------------------

        context_mask, target_mask = masker(
            batch.shape[0],
            device
        )

        # ----------------------------------------------------
        # Forward pass
        # ----------------------------------------------------

        predictions, targets = tjepa(
            batch,
            context_mask,
            target_mask
        )

        # ----------------------------------------------------
        # Loss
        # ----------------------------------------------------

        loss = tjepa_loss(
            predictions,
            targets,
            target_mask
        )

        # ----------------------------------------------------
        # Safety check
        # ----------------------------------------------------

        if not torch.isfinite(loss):

            raise RuntimeError(
                f"Non-finite loss detected at "
                f"epoch {epoch + 1}: {loss.item()}"
            )

        # ----------------------------------------------------
        # Backpropagation
        # ----------------------------------------------------

        optimizer_tj.zero_grad(
            set_to_none=True
        )

        loss.backward()

        # ----------------------------------------------------
        # Gradient clipping
        # ----------------------------------------------------

        torch.nn.utils.clip_grad_norm_(
            list(
                tjepa.context_encoder.parameters()
            )
            +
            list(
                tjepa.predictor.parameters()
            ),
            max_norm=1.0
        )

        optimizer_tj.step()

        # ----------------------------------------------------
        # EMA target encoder update
        # ----------------------------------------------------

        with torch.no_grad():

            for target_param, context_param in zip(
                tjepa.target_encoder.parameters(),
                tjepa.context_encoder.parameters()
            ):

                target_param.data.mul_(
                    current_ema
                )

                target_param.data.add_(
                    context_param.data,
                    alpha=1.0 - current_ema
                )

        epoch_loss += loss.item()
        batches += 1


    # --------------------------------------------------------
    # Epoch statistics
    # --------------------------------------------------------

    epoch_loss /= batches

    scheduler_tj.step()

    current_lr = optimizer_tj.param_groups[0]["lr"]

    training_history_tj.append({
        "epoch": epoch + 1,
        "loss": epoch_loss,
        "lr": current_lr,
        "ema": current_ema
    })


    # --------------------------------------------------------
    # Save best model
    # --------------------------------------------------------

    if epoch_loss < best_loss:

        best_loss = epoch_loss

        best_state_tj = {
            key: value.detach().cpu().clone()
            for key, value in tjepa.state_dict().items()
        }


    print(
        f"Epoch {epoch + 1:02d}/{TJ_EPOCHS} "
        f"| Loss: {epoch_loss:.6f} "
        f"| LR: {current_lr:.2e} "
        f"| EMA: {current_ema:.5f}"
    )

#### Restore best model

In [ ]:
if best_state_tj is not None:

    tjepa.load_state_dict(
        best_state_tj
    )

    tjepa.to(device)

In [ ]:
elapsed = time.time() - start_time

In [ ]:
print("\n" + "=" * 70)
print("T-JEPA PRETRAINING COMPLETE")
print("=" * 70)

print(
    f"Best training loss: {best_loss:.6f}"
)

print(
    f"Training time: {elapsed / 60:.2f} minutes"
)

### Step 16: Extract T-JEPA Test Anomaly Scores

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

# Inference-time anomaly scoring for T-JEPA.
# Masks are generated fresh per batch (stochastic), consistent with how
# the model was trained. Anomaly score = mean cosine error over target features.
tjepa.eval()

test_scores_tj = []
test_charge_ids_tj = []

print("=" * 70)
print("GENERATING T-JEPA TEST SCORES")
print("=" * 70)

with torch.no_grad():

    for batch in test_loader_tj:

        batch = batch.to(
            device,
            non_blocking=True
        )

        batch_size = batch.shape[0]

        # ----------------------------------------------------
        # Generate fresh masks
        # ----------------------------------------------------

        context_mask, target_mask = masker(
            batch_size,
            device
        )

        # ----------------------------------------------------
        # Forward pass
        # ----------------------------------------------------

        predictions, targets = tjepa(
            batch,
            context_mask,
            target_mask
        )

        # ----------------------------------------------------
        # Normalize latent vectors
        # ----------------------------------------------------

        pred_norm = F.normalize(
            predictions,
            dim=-1
        )

        target_norm = F.normalize(
            targets,
            dim=-1
        )

        # ----------------------------------------------------
        # Feature-level prediction error
        # ----------------------------------------------------

        feature_error = 1.0 - (
            pred_norm * target_norm
        ).sum(dim=-1)

        # ----------------------------------------------------
        # Only evaluate target features
        # ----------------------------------------------------

        feature_error = feature_error * target_mask.float()

        target_count = target_mask.sum(
            dim=1
        ).clamp(min=1)

        # Average target-feature error
        sample_score = (
            feature_error.sum(dim=1)
            /
            target_count
        )

        test_scores_tj.extend(
            sample_score.detach()
            .cpu()
            .numpy()
            .tolist()
        )

#### Convert to numpy

In [ ]:
test_scores_tj = np.asarray(
    test_scores_tj,
    dtype=np.float64
)

print("\n" + "=" * 70)
print("T-JEPA SCORING COMPLETE")
print("=" * 70)

print(
    "Number of scores:",
    len(test_scores_tj)
)

print(
    "Expected test rows:",
    len(test_df_fj)
)

print(
    "Mean score:",
    test_scores_tj.mean()
)

print(
    "Std score:",
    test_scores_tj.std()
)

print(
    "Min score:",
    test_scores_tj.min()
)

print(
    "Max score:",
    test_scores_tj.max()
)

print(
    "NaN:",
    np.isnan(test_scores_tj).sum()
)

print(
    "Inf:",
    np.isinf(test_scores_tj).sum()
)

### Step 17: T-JEPA Overall Fraud Evaluation

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    confusion_matrix
)

# ------------------------------------------------------------
# Build test evaluation dataframe
# ------------------------------------------------------------

# Build evaluation dataframe: one row per test transaction with tjepa_score,
# is_fraud label, and typology from the ground-truth file.
tj_eval = test_df_fj[
    ["id"]
].copy()

tj_eval["tjepa_score"] = test_scores_tj

# ------------------------------------------------------------
# Attach complete ground truth
# ------------------------------------------------------------

gt_eval = ground_truth[
    ["charge_id", "is_fraud", "typology"]
].copy()

gt_eval = gt_eval.rename(
    columns={
        "charge_id": "id"
    }
)

tj_eval = tj_eval.merge(
    gt_eval,
    on="id",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

print("=" * 70)
print("T-JEPA EVALUATION DATA")
print("=" * 70)

print("Rows:", len(tj_eval))

print(
    "Missing fraud labels:",
    tj_eval["is_fraud"].isna().sum()
)

print(
    "Missing typology:",
    tj_eval["typology"].isna().sum()
)

print(
    "Fraud count:",
    tj_eval["is_fraud"].sum()
)

print(
    "Normal count:",
    (tj_eval["is_fraud"] == 0).sum()
)

# ------------------------------------------------------------
# Overall metrics
# ------------------------------------------------------------

y_true = tj_eval["is_fraud"].astype(int).values
y_score = tj_eval["tjepa_score"].values

roc_auc_tj = roc_auc_score(
    y_true,
    y_score
)

pr_auc_tj = average_precision_score(
    y_true,
    y_score
)

print("\n" + "=" * 70)
print("T-JEPA OVERALL RESULTS")
print("=" * 70)

print(
    f"ROC-AUC : {roc_auc_tj:.4f}"
)

print(
    f"PR-AUC  : {pr_auc_tj:.4f}"
)

print(
    f"Fraud prevalence: {y_true.mean():.4%}"
)

### Step 18: T-JEPA Precision@k / Recall@k

In [ ]:
def precision_recall_at_k(y_true, scores, fraction):
    """
    Select the top fraction of samples by anomaly score.
    """

    n = len(y_true)

    k = max(
        1,
        int(np.ceil(n * fraction))
    )

    order = np.argsort(
        -scores
    )

    top_k = order[:k]

    selected_fraud = y_true[top_k].sum()

    precision = selected_fraud / k

    recall = selected_fraud / y_true.sum()

    return k, selected_fraud, precision, recall

In [ ]:
k_fractions = [
    0.001,   # 0.1%
    0.005,   # 0.5%
    0.01,    # 1%
    0.02,    # 2%
    0.05,    # 5%
    0.10     # 10%
]

results_prk_tj = []

for fraction in k_fractions:

    k, frauds, precision, recall = (
        precision_recall_at_k(
            y_true,
            y_score,
            fraction
        )
    )

    results_prk_tj.append({
        "Top_K_%": fraction * 100,
        "K": k,
        "Frauds_Found": frauds,
        "Precision": precision,
        "Recall": recall
    })

In [ ]:
prk_tj_df = pd.DataFrame(
    results_prk_tj
)

print("=" * 70)
print("T-JEPA PRECISION@K / RECALL@K")
print("=" * 70)

display(
    prk_tj_df.style.format({
        "Top_K_%": "{:.1f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}"
    })
)

### Step 19: T-JEPA Precision at Fixed Recall Levels

In [ ]:
# Find the smallest top-K set of flagged transactions whose recall
# reaches the requested target. Returns K, fraud count, precision, actual recall.
def precision_at_target_recall(
    y_true,
    scores,
    target_recall
):
    """
    Find the smallest top-K set whose recall
    reaches the requested target.
    """

    order = np.argsort(
        -scores
    )

    sorted_y = y_true[order]

    cumulative_fraud = np.cumsum(
        sorted_y
    )

    total_fraud = y_true.sum()

    recall_curve = (
        cumulative_fraud
        /
        total_fraud
    )

    idx = np.searchsorted(
        recall_curve,
        target_recall,
        side="left"
    )

    # Safety
    idx = min(
        idx,
        len(y_true) - 1
    )

    k = idx + 1

    frauds_found = cumulative_fraud[idx]

    precision = frauds_found / k

    actual_recall = (
        frauds_found
        /
        total_fraud
    )

    return (
        k,
        frauds_found,
        precision,
        actual_recall
    )

In [ ]:
recall_targets = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]

In [ ]:
fixed_recall_results_tj = []

In [ ]:
for target in recall_targets:

    (
        k,
        frauds_found,
        precision,
        actual_recall
    ) = precision_at_target_recall(
        y_true,
        y_score,
        target
    )

    fixed_recall_results_tj.append({

        "Target_Recall": target,

        "K": k,

        "Frauds_Found": frauds_found,

        "Precision": precision,

        "Actual_Recall": actual_recall
    })

In [ ]:
fixed_recall_tj_df = pd.DataFrame(
    fixed_recall_results_tj
)

In [ ]:
print("=" * 70)
print("T-JEPA PRECISION AT FIXED RECALL")
print("=" * 70)

In [ ]:
display(
    fixed_recall_tj_df.style.format({

        "Target_Recall": "{:.0%}",

        "Precision": "{:.4f}",

        "Actual_Recall": "{:.4f}"
    })
)

### Step 20: T-JEPA Typology-Wise Evaluation

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

print("=" * 70)
print("T-JEPA TYPOLOGY-WISE EVALUATION")
print("=" * 70)

typologies = [
    "normal",
    "ring",
    "ato",
    "emerge"
]

typology_results_tj = []

In [ ]:
for typology in typologies:

    # --------------------------------------------------------
    # Select this typology
    # --------------------------------------------------------

    subset = tj_eval[
        tj_eval["typology"] == typology
    ].copy()

    y_typ = subset[
        "is_fraud"
    ].astype(int).values

    scores_typ = subset[
        "tjepa_score"
    ].values

    n_samples = len(subset)

    n_fraud = int(
        y_typ.sum()
    )

    n_normal = (
        n_samples - n_fraud
    )

    prevalence = (
        n_fraud / n_samples
        if n_samples > 0
        else np.nan
    )

    # --------------------------------------------------------
    # Metrics require both classes
    # --------------------------------------------------------

    if (
        n_fraud > 0
        and
        n_normal > 0
    ):

        roc_auc = roc_auc_score(
            y_typ,
            scores_typ
        )

        pr_auc = average_precision_score(
            y_typ,
            scores_typ
        )

    else:

        roc_auc = np.nan
        pr_auc = np.nan


    typology_results_tj.append({

        "Typology": typology,

        "Samples": n_samples,

        "Fraud": n_fraud,

        "Normal": n_normal,

        "Fraud_Prevalence": prevalence,

        "ROC_AUC": roc_auc,

        "PR_AUC": pr_auc
    })

In [ ]:
typology_tj_df = pd.DataFrame(
    typology_results_tj
)

In [ ]:
print()

display(
    typology_tj_df.style.format({

        "Fraud_Prevalence": "{:.4%}",

        "ROC_AUC": "{:.4f}",

        "PR_AUC": "{:.4f}"
    })
)

### Step 21: Align T-JEPA with Fin-JEPA Test Targets

In [ ]:
# Fin-JEPA target charge IDs from the existing notebook
fin_ids = pd.Series(
    test_charge_ids
).astype(str)

# T-JEPA evaluation IDs
tj_ids = tj_eval["id"].astype(str)

common_ids = set(fin_ids) & set(tj_ids)

print("=" * 70)
print("COMMON EVALUATION SET")
print("=" * 70)

print("Fin-JEPA target IDs :", len(fin_ids))
print("T-JEPA test IDs     :", len(tj_ids))
print("Common IDs          :", len(common_ids))

print(
    "Fin-JEPA IDs unique :",
    fin_ids.nunique()
)

print(
    "T-JEPA IDs unique   :",
    tj_ids.nunique()
)

### Step 22: Common Evaluation Set

In [ ]:
# Restrict T-JEPA evaluation to exactly the 9,632 transactions
# that Fin-JEPA also evaluated (its sliding-window targets).
# This ensures both models are compared on an identical test set.
# ------------------------------------------------------------
# Make sure IDs have the same type
# ------------------------------------------------------------

tj_eval["id"] = tj_eval["id"].astype(str)

fin_ids = pd.Series(
    test_charge_ids
).astype(str)

# ------------------------------------------------------------
# Select exactly the Fin-JEPA target transactions
# ------------------------------------------------------------

common_eval = tj_eval[
    tj_eval["id"].isin(fin_ids)
].copy()

# Preserve exactly the Fin-JEPA target-ID order
fin_id_order = {
    charge_id: i
    for i, charge_id in enumerate(fin_ids)
}

common_eval["_fin_order"] = (
    common_eval["id"].map(fin_id_order)
)

common_eval = common_eval.sort_values(
    "_fin_order"
).reset_index(drop=True)

common_eval = common_eval.drop(
    columns=["_fin_order"]
)

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

print("=" * 70)
print("COMMON EVALUATION SET")
print("=" * 70)

print(
    "Rows:",
    len(common_eval)
)

print(
    "Unique IDs:",
    common_eval["id"].nunique()
)

print(
    "Fraud:",
    int(common_eval["is_fraud"].sum())
)

print(
    "Normal:",
    int((common_eval["is_fraud"] == 0).sum())
)

print(
    "Fraud prevalence:",
    f"{common_eval['is_fraud'].mean():.4%}"
)

print("\nFirst 5 IDs:")
print(
    common_eval[
        ["id", "is_fraud", "typology", "tjepa_score"]
    ].head()
)

### Step 24: T-JEPA — Common 9,632 Set Metrics

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

y_common = common_eval["is_fraud"].astype(int).values
scores_common = common_eval["tjepa_score"].values

# ------------------------------------------------------------
# Overall metrics
# ------------------------------------------------------------

roc_auc = roc_auc_score(
    y_common,
    scores_common
)

pr_auc = average_precision_score(
    y_common,
    scores_common
)

print("=" * 70)
print("T-JEPA — COMMON 9,632 TEST SET")
print("=" * 70)

print(f"ROC-AUC         : {roc_auc:.4f}")
print(f"PR-AUC          : {pr_auc:.4f}")
print(f"Fraud count     : {y_common.sum()}")
print(f"Normal count    : {(y_common == 0).sum()}")
print(f"Fraud prevalence: {y_common.mean():.4%}")

#### Precision@K / Recall@K

In [ ]:
print("\n" + "=" * 70)
print("PRECISION@K / RECALL@K")
print("=" * 70)

ranking = np.argsort(-scores_common)

fraud_total = y_common.sum()

for pct in [0.1, 0.5, 1, 2, 5, 10]:

    k = max(1, int(len(y_common) * pct / 100))

    top_k = ranking[:k]

    fraud_found = y_common[top_k].sum()

    precision = fraud_found / k
    recall = fraud_found / fraud_total

    print(
        f"Top {pct:>4}% | "
        f"K={k:>4} | "
        f"Frauds={fraud_found:>3} | "
        f"Precision={precision:.4f} | "
        f"Recall={recall:.4f}"
    )

### Step 25: Precision at Fixed Recall

In [ ]:
print("=" * 70)
print("PRECISION AT FIXED RECALL — T-JEPA")
print("=" * 70)

ranking = np.argsort(-scores_common)

for target_recall in [
    0.10, 0.20, 0.30, 0.40,
    0.50, 0.60, 0.70, 0.80, 0.90
]:

    required_frauds = int(
        np.ceil(target_recall * fraud_total)
    )

    cumulative_frauds = np.cumsum(
        y_common[ranking]
    )

    idx = np.where(
        cumulative_frauds >= required_frauds
    )[0]

    if len(idx) == 0:
        print(
            f"Recall {target_recall:.0%}: "
            "not achievable"
        )
        continue

    k = idx[0] + 1

    fraud_found = cumulative_frauds[idx[0]]

    precision = fraud_found / k
    actual_recall = fraud_found / fraud_total

    print(
        f"Target Recall={target_recall:.0%} | "
        f"K={k:>4} | "
        f"Frauds={fraud_found:>3} | "
        f"Precision={precision:.4f} | "
        f"Actual Recall={actual_recall:.4f}"
    )

### Step 27A: Find Fin-JEPA Score Variable — Safe Version

In [ ]:
print("=" * 70)
print("POSSIBLE FIN-JEPA SCORE VARIABLES")
print("=" * 70)

for name in list(globals().keys()):

    if "score" not in name.lower():
        continue

    try:
        value = globals()[name]

        if hasattr(value, "__len__"):
            print(
                f"{name:35s} "
                f"type={type(value).__name__:15s} "
                f"shape={getattr(value, 'shape', 'N/A')} "
                f"len={len(value)}"
            )

    except Exception as e:
        print(f"{name}: could not inspect")

### Step 27: Align Fin-JEPA and T-JEPA Scores

In [ ]:
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Merge Fin-JEPA (sequence-level) and T-JEPA (feature-level) scores onto
# the same 9,632-transaction common set so both can be compared fairly.
# Start from the common 9,632 transaction evaluation set
# ------------------------------------------------------------

comparison = common_eval[
    ["id", "is_fraud", "typology"]
].copy()

comparison["is_fraud"] = (
    comparison["is_fraud"]
    .astype(int)
)

# ------------------------------------------------------------
# T-JEPA score
# ------------------------------------------------------------

comparison["tjepa_score"] = scores_common

# ------------------------------------------------------------
# Fin-JEPA score
# ------------------------------------------------------------

comparison["finjepa_score"] = test_scores

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

print("=" * 70)
print("FIN-JEPA vs T-JEPA — COMMON SET")
print("=" * 70)

print("Rows:", len(comparison))

print(
    "Unique IDs:",
    comparison["id"].nunique()
)

print(
    "Fraud:",
    comparison["is_fraud"].sum()
)

print(
    "Normal:",
    (comparison["is_fraud"] == 0).sum()
)

print(
    "Fin-JEPA NaN:",
    comparison["finjepa_score"].isna().sum()
)

print(
    "T-JEPA NaN:",
    comparison["tjepa_score"].isna().sum()
)

print("\nFirst 5 rows:")
print(comparison.head())

### Step 28: Typology-vs-Normal Evaluation

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

# Evaluate each fraud typology against a clean normal baseline.
# Using normal non-fraud as the negative class isolates how well each model
# distinguishes each attack type from clean traffic.
typologies = ["ring", "ato", "emerge"]

results = []

# Normal, confirmed non-fraud transactions
normal_negative = comparison[
    (comparison["typology"] == "normal") &
    (comparison["is_fraud"] == 0)
].copy()

print("=" * 70)
print("TYPOLOGY-VS-NORMAL EVALUATION")
print("=" * 70)

print(
    f"Normal non-fraud reference: {len(normal_negative)}"
)

for typology in typologies:

    # Fraud transactions belonging to this typology
    fraud_typology = comparison[
        (comparison["typology"] == typology) &
        (comparison["is_fraud"] == 1)
    ].copy()

    print("\n" + "-" * 70)
    print(typology.upper())

    print(
        f"Fraud transactions: {len(fraud_typology)}"
    )

    # --------------------------------------------------------
    # If there are no frauds, metrics are undefined
    # --------------------------------------------------------

    if len(fraud_typology) == 0:

        print("No fraud transactions in common test set.")
        print("ROC-AUC: N/A")
        print("PR-AUC : N/A")

        results.append({
            "typology": typology,
            "fraud_count": 0,
            "normal_count": len(normal_negative),
            "finjepa_roc_auc": np.nan,
            "finjepa_pr_auc": np.nan,
            "tjepa_roc_auc": np.nan,
            "tjepa_pr_auc": np.nan
        })

        continue

    # --------------------------------------------------------
    # Typology fraud + normal non-fraud reference
    # --------------------------------------------------------

    eval_df = pd.concat(
        [
            fraud_typology,
            normal_negative
        ],
        ignore_index=True
    )

    y = eval_df["is_fraud"].astype(int).values

    # --------------------------------------------------------
    # Fin-JEPA
    # --------------------------------------------------------

    fin_scores = eval_df[
        "finjepa_score"
    ].values

    fin_roc = roc_auc_score(
        y,
        fin_scores
    )

    fin_pr = average_precision_score(
        y,
        fin_scores
    )

    # --------------------------------------------------------
    # T-JEPA
    # --------------------------------------------------------

    tj_scores = eval_df[
        "tjepa_score"
    ].values

    tj_roc = roc_auc_score(
        y,
        tj_scores
    )

    tj_pr = average_precision_score(
        y,
        tj_scores
    )

    print(
        f"Normal reference: {len(normal_negative)}"
    )

    print(
        f"Fin-JEPA ROC-AUC: {fin_roc:.4f}"
    )

    print(
        f"Fin-JEPA PR-AUC : {fin_pr:.4f}"
    )

    print(
        f"T-JEPA ROC-AUC  : {tj_roc:.4f}"
    )

    print(
        f"T-JEPA PR-AUC   : {tj_pr:.4f}"
    )

    results.append({
        "typology": typology,
        "fraud_count": len(fraud_typology),
        "normal_count": len(normal_negative),
        "finjepa_roc_auc": fin_roc,
        "finjepa_pr_auc": fin_pr,
        "tjepa_roc_auc": tj_roc,
        "tjepa_pr_auc": tj_pr
    })

In [ ]:
typology_results = pd.DataFrame(results)

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

display(typology_results)

### Step 29: Typology Precision@k / Recall@k

In [ ]:
import numpy as np
import pandas as pd

typologies = ["ring", "ato", "emerge"]

top_k_percentages = [
    0.1,
    0.5,
    1,
    2,
    5,
    10
]

typology_pk_results = []

In [ ]:
for typology in typologies:

    # --------------------------------------------------------
    # Fraud of this typology
    # --------------------------------------------------------

    fraud_typology = comparison[
        (comparison["typology"] == typology) &
        (comparison["is_fraud"] == 1)
    ].copy()

    # --------------------------------------------------------
    # Normal non-fraud reference
    # --------------------------------------------------------

    normal_negative = comparison[
        (comparison["typology"] == "normal") &
        (comparison["is_fraud"] == 0)
    ].copy()

    # --------------------------------------------------------
    # Skip typology if no fraud exists
    # --------------------------------------------------------

    if len(fraud_typology) == 0:

        print(
            f"\n{typology.upper()}: N/A "
            f"(no fraud transactions)"
        )

        continue

    # --------------------------------------------------------
    # Create typology-vs-normal evaluation set
    # --------------------------------------------------------

    eval_df = pd.concat(
        [
            fraud_typology,
            normal_negative
        ],
        ignore_index=True
    )

    y = eval_df["is_fraud"].astype(int).values

    fraud_total = y.sum()

    print("\n" + "=" * 70)
    print(f"{typology.upper()} — PRECISION@K / RECALL@K")
    print("=" * 70)

    print(
        f"Fraud count : {fraud_total}"
    )

    print(
        f"Normal count: {len(normal_negative)}"
    )

    # --------------------------------------------------------
    # Evaluate both models
    # --------------------------------------------------------

    for model_name, score_column in [
        ("Fin-JEPA", "finjepa_score"),
        ("T-JEPA", "tjepa_score")
    ]:

        scores = eval_df[
            score_column
        ].values

        ranking = np.argsort(-scores)

        print(
            f"\n--- {model_name} ---"
        )

        for pct in top_k_percentages:

            k = max(
                1,
                int(len(eval_df) * pct / 100)
            )

            top_k = ranking[:k]

            fraud_found = y[top_k].sum()

            precision = fraud_found / k

            recall = (
                fraud_found / fraud_total
            )

            print(
                f"Top {pct:>4}% | "
                f"K={k:>4} | "
                f"Frauds={fraud_found:>3} | "
                f"Precision={precision:.4f} | "
                f"Recall={recall:.4f}"
            )

            typology_pk_results.append({
                "typology": typology,
                "model": model_name,
                "top_percent": pct,
                "K": k,
                "fraud_found": int(fraud_found),
                "precision": precision,
                "recall": recall
            })

In [ ]:
typology_pk_results = pd.DataFrame(
    typology_pk_results
)

print("\n" + "=" * 70)
print("TYPOLOGY PRECISION@K / RECALL@K SUMMARY")
print("=" * 70)

display(typology_pk_results)

## Part 5: Typology and failure analysis

Examine the emerge typology and study which fraud cases both models miss.

### Emerge typology: train vs test distribution

In [ ]:
# Check typology distribution in TRAIN and TEST
print("TRAIN TYPOLOGY")
print(
    train_df_fj.groupby("typology")["is_fraud"]
    .agg(["count", "sum"])
)

print("\nTEST TYPOLOGY")
print(
    test_df_fj.groupby("typology")["is_fraud"]
    .agg(["count", "sum"])
)

In [ ]:
train_emerge = train_df_fj[
    (train_df_fj["typology"] == "emerge") &
    (train_df_fj["is_fraud"] == 1)
]

test_emerge = test_df_fj[
    (test_df_fj["typology"] == "emerge") &
    (test_df_fj["is_fraud"] == 1)
]

print("Train emerge fraud:", len(train_emerge))
print("Test emerge fraud:", len(test_emerge))

In [ ]:
for split_name, split_df in [
    ("TRAIN", train_df_fj),
    ("TEST", test_df_fj)
]:
    x = split_df[
        (split_df["typology"] == "emerge") &
        (split_df["is_fraud"] == 1)
    ]

    print(f"\n{split_name} emerge fraud")
    print("Count:", len(x))
    print("First:", x["created"].min())
    print("Last :", x["created"].max())

In [ ]:
print("Overall emerge fraud timeline:")

emerge_fraud = df_full[
    (df_full["typology"] == "emerge") &
    (df_full["is_fraud"] == 1)
]

print("Count:", len(emerge_fraud))
print("First:", emerge_fraud["created"].min())
print("Last :", emerge_fraud["created"].max())

In [ ]:
train_emerge = train_df_fj[
    train_df_fj["typology"] == "emerge"
]

test_emerge = test_df_fj[
    test_df_fj["typology"] == "emerge"
]

for col in ["customer_id", "card_id", "connected_account_id"]:
    train_ids = set(train_emerge[col].dropna())
    test_ids = set(test_emerge[col].dropna())

    overlap = train_ids & test_ids

    print(f"\n{col}")
    print("Train unique:", len(train_ids))
    print("Test unique :", len(test_ids))
    print("Overlap     :", len(overlap))

### Check available score variables

In [ ]:
for name in [
    "scores_common",
    "test_scores",
    "test_charge_ids",
    "ground_truth",
    "gt",
    "comparison_df"
]:
    print(name, "->", name in globals())

### Failure analysis: build combined comparison table

In [ ]:
import pandas as pd
import numpy as np

# Build a single comparison dataframe aligning Fin-JEPA and T-JEPA scores
# with ground-truth labels for the common 9,632-transaction test set.
# Ground truth
gt_df = ground_truth.copy()

print("Ground truth shape:", gt_df.shape)
print("Ground truth columns:", gt_df.columns.tolist())

# Make sure the ID column is named consistently
gt_df = gt_df.rename(columns={"charge_id": "id"})

# Create Fin-JEPA dataframe
fin_df = pd.DataFrame({
    "id": test_charge_ids,
    "finjepa_score": test_scores
})

# Create T-JEPA dataframe
# scores_common is already aligned to the same 9,632 common IDs
tjepa_df = pd.DataFrame({
    "id": test_charge_ids,
    "tjepa_score": scores_common
})

# Merge the scores
comparison_df = fin_df.merge(
    tjepa_df,
    on="id",
    how="inner"
)

# Add ground-truth fraud label and typology
comparison_df = comparison_df.merge(
    gt_df[["id", "is_fraud", "typology"]],
    on="id",
    how="left"
)

print("\nComparison shape:", comparison_df.shape)
print("\nFraud count:", comparison_df["is_fraud"].sum())
print("Normal count:", (comparison_df["is_fraud"] == 0).sum())
print("Missing labels:", comparison_df["is_fraud"].isna().sum())
print("Missing typology:", comparison_df["typology"].isna().sum())

print("\nFirst 5 rows:")
display(comparison_df.head())

#### Rank scores and flag top-1% detections

In [ ]:
# Rank all test transactions by anomaly score (rank 1 = most anomalous).
# 'top1' flag marks transactions in the top 1% — operationally the alert list.
# Create a copy so we don't modify the original
failure_df = comparison_df.copy()

# Rank 1 = highest anomaly score
failure_df["fin_rank"] = (
    failure_df["finjepa_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

failure_df["tjepa_rank"] = (
    failure_df["tjepa_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

print("Shape:", failure_df.shape)

print("\nColumns:")
print(failure_df.columns.tolist())

print("\nRank range:")
print("Fin-JEPA:", failure_df["fin_rank"].min(), 
      "to", failure_df["fin_rank"].max())
print("T-JEPA:", failure_df["tjepa_rank"].min(), 
      "to", failure_df["tjepa_rank"].max())

print("\nTop 10 by Fin-JEPA:")
display(
    failure_df.sort_values("fin_rank").head(10)
)

print("\nTop 10 by T-JEPA:")
display(
    failure_df.sort_values("tjepa_rank").head(10)
)

#### Group fraud cases by which model detected them

In [ ]:
# Get only the 107 ground-truth fraud cases
fraud_cases = failure_df[
    failure_df["is_fraud"] == True
].copy()

print("Total fraud cases:", len(fraud_cases))

# Top 1% and Top 5%
N = len(failure_df)

k1 = int(np.ceil(0.01 * N))
k5 = int(np.ceil(0.05 * N))

print("Top 1% K =", k1)
print("Top 5% K =", k5)

# Whether each fraud was detected by each model
fraud_cases["fin_top1"] = fraud_cases["fin_rank"] <= k1
fraud_cases["tjepa_top1"] = fraud_cases["tjepa_rank"] <= k1

fraud_cases["fin_top5"] = fraud_cases["fin_rank"] <= k5
fraud_cases["tjepa_top5"] = fraud_cases["tjepa_rank"] <= k5

In [ ]:
# Classification at Top 1%
def classify_top1(row):
    fin = row["fin_top1"]
    tj = row["tjepa_top1"]

    if fin and tj:
        return "Both"
    elif fin and not tj:
        return "Fin-JEPA only"
    elif not fin and tj:
        return "T-JEPA only"
    else:
        return "Both missed"

In [ ]:
fraud_cases["group_top1"] = fraud_cases.apply(
    classify_top1,
    axis=1
)

print("\n=== TOP 1% FAILURE GROUPS ===")
print(fraud_cases["group_top1"].value_counts())

print("\n=== TOP 1% BY TYPOLOGY ===")
print(
    pd.crosstab(
        fraud_cases["group_top1"],
        fraud_cases["typology"]
    )
)

In [ ]:
# Create a simple failure category
fraud_cases["failure_category"] = np.where(
    fraud_cases["group_top1"] == "Both missed",
    "Both missed",
    "Detected by at least one"
)

print(
    fraud_cases["failure_category"].value_counts()
)

print("\nTypology distribution:")
print(
    pd.crosstab(
        fraud_cases["failure_category"],
        fraud_cases["typology"]
    )
)

In [ ]:
both_missed = fraud_cases[
    fraud_cases["group_top1"] == "Both missed"
].copy()

print("Both-missed fraud cases:", len(both_missed))

display(
    both_missed[
        [
            "id",
            "typology",
            "finjepa_score",
            "fin_rank",
            "tjepa_score",
            "tjepa_rank"
        ]
    ]
    .sort_values("fin_rank", ascending=False)
    .head(30)
)

In [ ]:
print("Original group_top1 values:")
print(fraud_cases["group_top1"].value_counts(dropna=False))

print("\nTypology:")
print(
    pd.crosstab(
        fraud_cases["group_top1"],
        fraud_cases["typology"],
        dropna=False
    )
)

In [ ]:
print("\nAll fraud cases by group:")
display(
    fraud_cases[
        [
            "id",
            "typology",
            "fin_rank",
            "tjepa_rank",
            "fin_top1",
            "tjepa_top1",
            "group_top1"
        ]
    ].sort_values("fin_rank")
)

#### Jointly missed vs detected fraud

In [ ]:
# Separate jointly missed frauds from frauds detected by at least one model

missed_both = fraud_cases[
    fraud_cases["group_top1"] == "Both missed"
].copy()

detected_by_any = fraud_cases[
    fraud_cases["group_top1"] != "Both missed"
].copy()

print("Both missed:", len(missed_both))
print("Detected by at least one:", len(detected_by_any))

print("\nTypology - Both missed:")
print(missed_both["typology"].value_counts())

print("\nTypology - Detected by at least one:")
print(detected_by_any["typology"].value_counts())

### Feature comparison: missed vs detected fraud

In [ ]:
print("df_full shape:")
print(df_full.shape)

print("\ndf_full columns:")
print(df_full.columns.tolist())

In [ ]:
# Attach raw/engineered transaction features to each fraud case so we can
# compare feature distributions between missed and detected frauds.
# Keep the fraud-case information we already calculated
fraud_analysis = fraud_cases.copy()

# Select transaction-level features from df_full
feature_cols = [
    "id",
    "amount",
    "currency",
    "captured",
    "card_brand",
    "card_funding",
    "card_country",
    "card_cvc_check",
    "cus_country",
    "cus_age_days",
    "mcc",
    "acct_country",
    "acct_structure",
    "geo_mismatch_cus",
    "geo_mismatch_card",
    "cvc_failed",
    "hour",
    "dow",
    "log_amount",
    "seq_pos",
    "secs_since_prev",
    "secs_since_first",
    "amount_prev",
    "amount_cummean",
    "amount_cumstd",
    "amount_z",
    "log_amount_prev",
    "log_amount_cummean",
    "log_amount_cumstd",
    "log_amount_z",
    "velocity_1h",
    "velocity_24h",
    "velocity_7d"
]

transaction_features = df_full[feature_cols].copy()

# Remove duplicate IDs if any
transaction_features = transaction_features.drop_duplicates("id")

# Merge features onto the 107 fraud cases
fraud_analysis = fraud_analysis.merge(
    transaction_features,
    on="id",
    how="left",
    validate="one_to_one"
)

print("Shape:", fraud_analysis.shape)
print("Missing feature rows:", fraud_analysis[feature_cols[1:]].isna().all(axis=1).sum())

print("\nColumns:")
print(fraud_analysis.columns.tolist())

In [ ]:
# Recreate the two failure groups

missed = fraud_analysis[
    fraud_analysis["group_top1"] == "Both missed"
].copy()

detected = fraud_analysis[
    fraud_analysis["group_top1"] != "Both missed"
].copy()

print("Both missed:", len(missed))
print("Detected by at least one:", len(detected))

In [ ]:
numeric_features = [
    "amount",
    "cus_age_days",
    "seq_pos",
    "secs_since_prev",
    "secs_since_first",
    "amount_prev",
    "amount_cummean",
    "amount_cumstd",
    "amount_z",
    "log_amount",
    "log_amount_prev",
    "log_amount_cummean",
    "log_amount_cumstd",
    "log_amount_z",
    "velocity_1h",
    "velocity_24h",
    "velocity_7d"
]

comparison = pd.DataFrame({
    "Both_missed_mean": missed[numeric_features].mean(),
    "Detected_mean": detected[numeric_features].mean(),
    "Both_missed_median": missed[numeric_features].median(),
    "Detected_median": detected[numeric_features].median()
})

comparison["mean_difference"] = (
    comparison["Detected_mean"] -
    comparison["Both_missed_mean"]
)

print(comparison.round(4))

In [ ]:
comparison_sorted = comparison.sort_values(
    "mean_difference",
    key=lambda x: x.abs(),
    ascending=False
)

print(comparison_sorted.round(4))

#### Emerge typology: missed vs detected

In [ ]:
emerge_missed = fraud_analysis[
    (fraud_analysis["typology"] == "emerge") &
    (fraud_analysis["group_top1"] == "Both missed")
].copy()

emerge_detected = fraud_analysis[
    (fraud_analysis["typology"] == "emerge") &
    (fraud_analysis["group_top1"] != "Both missed")
].copy()

print("Emerge missed:", len(emerge_missed))
print("Emerge detected:", len(emerge_detected))

In [ ]:
emerge_comparison = pd.DataFrame({
    "Missed_mean": emerge_missed[numeric_features].mean(),
    "Detected_mean": emerge_detected[numeric_features].mean(),
    "Missed_median": emerge_missed[numeric_features].median(),
    "Detected_median": emerge_detected[numeric_features].median()
})

emerge_comparison["mean_difference"] = (
    emerge_comparison["Detected_mean"] -
    emerge_comparison["Missed_mean"]
)

emerge_comparison = emerge_comparison.sort_values(
    "mean_difference",
    key=lambda x: x.abs(),
    ascending=False
)

print(emerge_comparison.round(4))

In [ ]:
# Compare failure groups by typology and sequence position

print("Sequence position by group:")
print(
    fraud_analysis.groupby("group_top1")["seq_pos"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(4)
)

print("\nVelocity features by group:")
print(
    fraud_analysis.groupby("group_top1")[
        ["velocity_1h", "velocity_24h", "velocity_7d"]
    ]
    .agg(["mean", "median"])
    .round(4)
)

In [ ]:
print("\nEmerge sequence position:")
print(
    fraud_analysis[
        fraud_analysis["typology"] == "emerge"
    ]
    .groupby("group_top1")["seq_pos"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(4)
)

### Diagnostics: inspect models, functions and masks

In [ ]:
import torch
import numpy as np
import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Deterministic seed set:", SEED)

In [ ]:
# Show functions currently defined in the notebook
[name for name in globals() if callable(globals()[name])]

In [ ]:
import inspect

print("TJEPA.forward:")
print(inspect.signature(tjepa.forward))

print("\nTJEPA class:")
print(inspect.signature(TJEPA))

print("\nFeatureMasker:")
print(inspect.signature(FeatureMasker))

print("\nTabularDataset:")
print(inspect.signature(TabularDataset))

In [ ]:
print("\nCurrent T-JEPA model:")
print(tjepa)

print("\nModel training mode:")
print(tjepa.training)

print("\nModel device:")
print(next(tjepa.parameters()).device)

In [ ]:
import inspect

print("TJEPA.forward signature:")
print(inspect.signature(TJEPA.forward))

print("\nTJEPA.forward source:")
try:
    print(inspect.getsource(TJEPA.forward))
except Exception as e:
    print("Could not get source:", e)

print("\nCurrent model forward test:")
print("test_ctx shape:", test_ctx.shape)
print("test_tgt shape:", test_tgt.shape)
print("test_ctx dtype:", test_ctx.dtype)
print("test_tgt dtype:", test_tgt.dtype)

In [ ]:
context_mask, target_mask = masker(test_ctx.shape[0], test_ctx.device)

print("context_mask shape:", context_mask.shape)
print("target_mask shape:", target_mask.shape)

print("context_mask dtype:", context_mask.dtype)
print("target_mask dtype:", target_mask.dtype)

print("Context mask values:", torch.unique(context_mask))
print("Target mask values:", torch.unique(target_mask))

print("Context feature counts, first 10 rows:")
print(context_mask.sum(dim=1)[:10])

print("Target feature counts, first 10 rows:")
print(target_mask.sum(dim=1)[:10])

In [ ]:
print("scores_common:")
print(scores_common[:10])
print("shape:", scores_common.shape)

In [ ]:
import inspect

# Show all currently defined functions that may be related to scoring
for name in [
    "precision_recall_at_k",
    "precision_at_target_recall",
    "classify_top1",
    "tjepa_loss"
]:
    obj = globals().get(name)
    print("\n" + "="*70)
    print(name, ":", obj)
    if obj is not None:
        try:
            print(inspect.signature(obj))
            print(inspect.getsource(obj))
        except Exception as e:
            print("Could not retrieve source:", e)

In [ ]:
print("Searching notebook history...\n")

keywords = [
    "scores_common",
    "test_scores",
    "target_tokens",
    "predictions",
    "tjepa(",
    "tjepa.forward",
]

for i, cell in enumerate(In):
    if any(k in cell for k in keywords):
        print("\n" + "="*80)
        print(f"CELL {i}")
        print("="*80)
        print(cell)

In [ ]:
import inspect

print("T-JEPA loss signature:")
print(inspect.signature(tjepa_loss))

print("\nT-JEPA loss source:")
try:
    print(inspect.getsource(tjepa_loss))
except Exception as e:
    print("Could not retrieve source:", e)

In [ ]:
print("T-JEPA object:")
print(tjepa)

print("\nT-JEPA forward:")
print(inspect.signature(tjepa.forward))

In [ ]:
print("scores_common shape:", scores_common.shape)

## Part 6: Deterministic T-JEPA scoring and feature sensitivity

Re-score T-JEPA with fixed masks, then measure feature sensitivity and compare missed vs detected fraud.

### Step 28A: Create Fixed Evaluation Masks

In [ ]:
import random
import torch
import numpy as np

# Important: fixed seed specifically for evaluation masks
EVAL_SEED = 12345

random.seed(EVAL_SEED)
torch.manual_seed(EVAL_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(EVAL_SEED)

# Number of test samples
N_TEST = len(X_test_np)

fixed_context_masks, fixed_target_masks = masker(
    N_TEST,
    device
)

print("=" * 70)
print("FIXED T-JEPA EVALUATION MASKS")
print("=" * 70)

print("Context mask shape:", fixed_context_masks.shape)
print("Target mask shape :", fixed_target_masks.shape)

print(
    "Average context features:",
    fixed_context_masks.sum(dim=1).float().mean().item()
)

print(
    "Average target features:",
    fixed_target_masks.sum(dim=1).float().mean().item()
)

print(
    "Any overlap:",
    (fixed_context_masks & fixed_target_masks).any().item()
)

### Step 28B: Deterministic T-JEPA Scoring Function

In [ ]:
import torch.nn.functional as F

# Deterministic scoring function: uses pre-generated fixed masks rather than
# sampling new masks per batch. This makes scores reproducible across runs,
# which is required for the feature perturbation sensitivity analysis below.
def score_tjepa_fixed(
    X_np,
    context_masks,
    target_masks,
    batch_size=256
):
    """
    Score T-JEPA using fixed masks.

    This follows the original Cell 22 scoring procedure:
      prediction/target normalization
      cosine feature error
      target-feature masking
      average target-feature error
    """

    tjepa.eval()

    scores = []

    X_tensor = torch.from_numpy(
        np.asarray(X_np, dtype=np.float32)
    )

    with torch.no_grad():

        for start in range(
            0,
            len(X_tensor),
            batch_size
        ):

            end = min(
                start + batch_size,
                len(X_tensor)
            )

            batch = X_tensor[
                start:end
            ].to(
                device,
                non_blocking=True
            )

            context_mask = context_masks[
                start:end
            ]

            target_mask = target_masks[
                start:end
            ]

            # --------------------------------------------
            # Forward
            # --------------------------------------------

            predictions, targets = tjepa(
                batch,
                context_mask,
                target_mask
            )

            # --------------------------------------------
            # Normalize latent vectors
            # --------------------------------------------

            pred_norm = F.normalize(
                predictions,
                dim=-1
            )

            target_norm = F.normalize(
                targets,
                dim=-1
            )

            # --------------------------------------------
            # Feature-level cosine error
            # --------------------------------------------

            feature_error = 1.0 - (
                pred_norm * target_norm
            ).sum(dim=-1)

            # --------------------------------------------
            # Only target features
            # --------------------------------------------

            feature_error = (
                feature_error
                * target_mask.float()
            )

            target_count = (
                target_mask
                .sum(dim=1)
                .clamp(min=1)
            )

            # --------------------------------------------
            # Transaction anomaly score
            # --------------------------------------------

            sample_score = (
                feature_error.sum(dim=1)
                / target_count
            )

            scores.extend(
                sample_score
                .detach()
                .cpu()
                .numpy()
                .tolist()
            )

    return np.asarray(
        scores,
        dtype=np.float64
    )

### Step 28C: Deterministic Baseline

In [ ]:
baseline_scores = score_tjepa_fixed(
    X_test_np,
    fixed_context_masks,
    fixed_target_masks
)

print("=" * 70)
print("DETERMINISTIC T-JEPA BASELINE")
print("=" * 70)

print("Number of scores:", len(baseline_scores))
print("Mean:", baseline_scores.mean())
print("Std :", baseline_scores.std())
print("Min :", baseline_scores.min())
print("Max :", baseline_scores.max())

print(
    "NaN:",
    np.isnan(baseline_scores).sum()
)

print(
    "Inf:",
    np.isinf(baseline_scores).sum()
)

print("\nFirst 10 scores:")
print(baseline_scores[:10])

In [ ]:
print("comparison_df:", len(comparison_df))
print("X_test_np:", len(X_test_np))
print("baseline_scores:", len(baseline_scores))

#### Get Row Indices of The Common 9,632 Ids in T-JEPA Test Set

In [ ]:
common_ids = comparison_df["id"].tolist()

tjepa_id_to_idx = {
    cid: i
    for i, cid in enumerate(test_charge_ids)
}

common_indices = [
    tjepa_id_to_idx[cid]
    for cid in common_ids
]

print("Common indices:", len(common_indices))
print("Unique indices:", len(set(common_indices)))

In [ ]:
X_common_np = X_test_np[common_indices]

print("X_common_np shape:", X_common_np.shape)

In [ ]:
EVAL_SEED = 12345

random.seed(EVAL_SEED)
np.random.seed(EVAL_SEED)
torch.manual_seed(EVAL_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(EVAL_SEED)

N_COMMON = len(X_common_np)

fixed_context_masks, fixed_target_masks = masker(
    N_COMMON,
    device
)

print("Context masks:", fixed_context_masks.shape)
print("Target masks:", fixed_target_masks.shape)

In [ ]:
baseline_scores = score_tjepa_fixed(
    X_common_np,
    fixed_context_masks,
    fixed_target_masks
)

print("Baseline scores:", len(baseline_scores))
print("Min:", baseline_scores.min())
print("Max:", baseline_scores.max())
print("Mean:", baseline_scores.mean())

### Step 28D: Baseline Fraud Check

In [ ]:
comparison_fixed = comparison_df.copy()

comparison_fixed["baseline_tjepa_score"] = baseline_scores

print(
    comparison_fixed.groupby("is_fraud")[
        "baseline_tjepa_score"
    ].agg(
        ["count", "mean", "median", "std", "min", "max"]
    )
)

### Step 28E: Define Feature Names

In [ ]:
feature_names = [
    "amount",
    "captured",
    "cus_age_days",
    "geo_mismatch_cus",
    "geo_mismatch_card",
    "cvc_failed",
    "hour",
    "dow",
    "log_amount",
    "seq_pos",
    "secs_since_prev",
    "secs_since_first",
    "amount_prev",
    "amount_cummean",
    "amount_cumstd",
    "amount_z",
    "log_amount_prev",
    "log_amount_cummean",
    "log_amount_cumstd",
    "log_amount_z",
    "velocity_1h",
    "velocity_24h",
    "velocity_7d",
    "currency",
    "billing_detail_address_country",
    "billing_detail_address_city",
    "card_brand",
    "card_funding",
    "card_country",
    "card_cvc_check",
    "cus_country",
    "cus_city",
    "mcc",
    "acct_country",
    "acct_structure"
]

# feature_baseline: per-feature value substituted when perturbing that feature.
# Using the training median approximates the 'most common' value seen during training.
# TODO(check): `feature_baseline` is used below but was never assigned in the
# notebook export (its defining cell was probably deleted or overwritten in the
# original session). Placeholder: per-feature median of the TRAIN array.
# Replace with your original definition if it differs.
if "feature_baseline" not in globals():
    feature_baseline = np.median(X_train_np, axis=0)

print("Number of feature names:", len(feature_names))
print("Number of feature baselines:", len(feature_baseline))

In [ ]:
for i, name in enumerate(feature_names):
    print(f"{i:2d} | {name:35s} | baseline = {feature_baseline[i]:.6f}")

### Step 28F: Feature Perturbation Sensitivity

In [ ]:
# Permutation-style sensitivity analysis: for each feature, replace its values
# with the training baseline and measure the mean absolute change in anomaly scores.
# Larger change -> the model relies more heavily on that feature.
importance_results = []

for feature_idx, feature_name in enumerate(feature_names):

    X_perturbed = X_common_np.copy()
    X_perturbed[:, feature_idx] = feature_baseline[feature_idx]

    perturbed_scores = score_tjepa_fixed(
        X_perturbed,
        fixed_context_masks,
        fixed_target_masks
    )

    score_change = perturbed_scores - baseline_scores

    importance_results.append({
        "feature": feature_name,
        "mean_abs_change": np.mean(np.abs(score_change)),
        "median_abs_change": np.median(np.abs(score_change)),
        "mean_signed_change": np.mean(score_change),
    })

importance_df = pd.DataFrame(importance_results)

importance_df = importance_df.sort_values(
    "mean_abs_change",
    ascending=False
).reset_index(drop=True)

print(importance_df)

### Step 28G: Jointly Missed Fraud Analysis

In [ ]:
fraud_mask = comparison_fixed["is_fraud"].to_numpy().astype(bool)

missed_mask = (
    fraud_mask &
    (failure_df["group_top1"].to_numpy() == "Both missed")
)

detected_fraud_mask = (
    fraud_mask &
    (failure_df["group_top1"].to_numpy() != "Both missed")
)

print("Total fraud:", fraud_mask.sum())
print("Jointly missed fraud:", missed_mask.sum())
print("Detected fraud:", detected_fraud_mask.sum())

In [ ]:
K = int(np.ceil(0.01 * len(comparison_fixed)))

print("K:", K)

In [ ]:
print(failure_df.columns.tolist())

### Step 28G: Recreate Failure Groups

In [ ]:
# Recompute detection flags and group assignments.
# Required here because failure_df may have been modified after Step 28G's first pass.
K = int(np.ceil(0.01 * len(comparison_fixed)))

failure_df["fin_top1"] = failure_df["fin_rank"] <= K
failure_df["tjepa_top1"] = failure_df["tjepa_rank"] <= K

failure_df["group_top1"] = np.select(
    [
        failure_df["fin_top1"] & failure_df["tjepa_top1"],
        failure_df["fin_top1"] & ~failure_df["tjepa_top1"],
        ~failure_df["fin_top1"] & failure_df["tjepa_top1"],
    ],
    [
        "Both",
        "Fin-JEPA only",
        "T-JEPA only",
    ],
    default="Both missed"
)

print(failure_df["group_top1"].value_counts())

In [ ]:
fraud_mask = failure_df["is_fraud"].to_numpy().astype(bool)

missed_mask = (
    fraud_mask &
    (failure_df["group_top1"].to_numpy() == "Both missed")
)

detected_fraud_mask = (
    fraud_mask &
    (failure_df["group_top1"].to_numpy() != "Both missed")
)

print("Total fraud:", fraud_mask.sum())
print("Jointly missed fraud:", missed_mask.sum())
print("Detected fraud:", detected_fraud_mask.sum())

### Step 28H: Missed vs Detected Fraud Feature Sensitivity

In [ ]:
# Feature sensitivity split by failure group.
# 'difference' > 0 means the feature matters more for detected fraud than for
# jointly-missed fraud, suggesting the missed cases have weaker signal on that feature.
failure_importance = []

for feature_idx, feature_name in enumerate(feature_names):

    X_perturbed = X_common_np.copy()

    X_perturbed[:, feature_idx] = feature_baseline[feature_idx]

    perturbed_scores = score_tjepa_fixed(
        X_perturbed,
        fixed_context_masks,
        fixed_target_masks
    )

    score_change = perturbed_scores - baseline_scores

    failure_importance.append({
        "feature": feature_name,
        "missed_mean_abs_change":
            np.mean(np.abs(score_change[missed_mask])),
        "detected_mean_abs_change":
            np.mean(np.abs(score_change[detected_fraud_mask])),
        "difference":
            np.mean(np.abs(score_change[detected_fraud_mask]))
            - np.mean(np.abs(score_change[missed_mask]))
    })

failure_importance_df = pd.DataFrame(failure_importance)

failure_importance_df = failure_importance_df.sort_values(
    "difference",
    ascending=False
).reset_index(drop=True)

print(failure_importance_df)

### Step 28I: Missed Fraud by Typology

In [ ]:
missed_fraud_df = failure_df[
    (failure_df["is_fraud"] == True) &
    (failure_df["group_top1"] == "Both missed")
].copy()

detected_fraud_df = failure_df[
    (failure_df["is_fraud"] == True) &
    (failure_df["group_top1"] != "Both missed")
].copy()

print("Missed fraud by typology:")
print(missed_fraud_df["typology"].value_counts())

print("\nDetected fraud by typology:")
print(detected_fraud_df["typology"].value_counts())

### Step 28J: Missed vs Detected Fraud Feature Values

In [ ]:
numeric_features = [
    "amount",
    "cus_age_days",
    "seq_pos",
    "secs_since_prev",
    "secs_since_first",
    "amount_prev",
    "amount_cummean",
    "amount_cumstd",
    "amount_z",
    "log_amount",
    "log_amount_prev",
    "log_amount_cummean",
    "log_amount_cumstd",
    "log_amount_z",
    "velocity_1h",
    "velocity_24h",
    "velocity_7d"
]

feature_value_results = []

for feature in numeric_features:

    missed_values = fraud_analysis.loc[
        fraud_analysis["id"].isin(missed_fraud_df["id"]),
        feature
    ]

    detected_values = fraud_analysis.loc[
        fraud_analysis["id"].isin(detected_fraud_df["id"]),
        feature
    ]

    feature_value_results.append({
        "feature": feature,
        "missed_mean": missed_values.mean(),
        "detected_mean": detected_values.mean(),
        "missed_median": missed_values.median(),
        "detected_median": detected_values.median(),
        "mean_difference": (
            detected_values.mean() - missed_values.mean()
        )
    })

feature_values_df = pd.DataFrame(feature_value_results)

feature_values_df = feature_values_df.sort_values(
    "mean_difference",
    key=lambda x: np.abs(x),
    ascending=False
).reset_index(drop=True)

print(feature_values_df)